# Over which fields is the Fermat quartic a Kummer surface?

The companion notebook [fermat-quartic-kummer.ipynb](fermat-quartic-kummer.ipynb) checks Mizukami's
isomorphism between the Fermat quartic surface
$$X:\; x_0^4+x_1^4+x_2^4+x_3^4 = 0 \;\subset\; \mathbb{P}^3$$
and the Kummer surface of the abelian surface $A \cong E_i \times E_{2i}$. Both $X$ and $\operatorname{Kum}(A)$
are defined by equations with **rational** coefficients, but the isomorphism between them is not: it needs
$k = \mathbb{Q}(\mu_8) = \mathbb{Q}(i,\sqrt2)$.

This notebook asks how much of that is really necessary, and finds that a good deal of it is not.

**Two dials we are allowed to turn.**

1. Ask for the isomorphism over a field strictly between $\mathbb{Q}$ and $\mathbb{Q}(\mu_8)$, rather than
   over $\mathbb{Q}$ itself. There are three such fields: $\mathbb{Q}(i)$, $\mathbb{Q}(\sqrt2)$,
   $\mathbb{Q}(\sqrt{-2})$.
2. Replace $X$ by a *twist* $a x_0^4 + b x_1^4 + c x_2^4 + d x_3^4 = 0$ — still a diagonal quartic, still
   the same surface after extending scalars — and correspondingly allow a different Kummer surface on the
   other side.

**The answers, obtained below.**

- Over $\mathbb{Q}(\sqrt2)$ and over $\mathbb{Q}(\sqrt{-2})$ the Fermat quartic **is** a Kummer surface.
  No twisting is needed at all; the field of definition of the isomorphism simply drops from degree 4 to
  degree 2 over $\mathbb{Q}$.
- Over $\mathbb{Q}(i)$ it is **not**, and this needs no search — a single rank count settles it.
- Over $\mathbb{Q}$ the Fermat quartic is not a Kummer surface, but the twist
  $x^4+y^4+4z^4+4w^4=0$ **is**.
- In every case found, the abelian surface on the other side is a nontrivial *torsor*, not an abelian
  surface. So $\operatorname{Kum}(E\times E')$ itself, with a genuine origin, still does not descend.

Everything is verified by machine below, twice over by two independent methods.

> **A note on what this notebook assumes.** Very little. Sections 1–4 explain from scratch what a Kummer
> surface is, what "twisting" means, and why the whole question turns into linear algebra over $\mathbb{Z}$.
> A reader who knows all that can start at section 5.

## 1. What a Kummer surface is

Start one dimension down, with an elliptic curve $E$ over $\mathbb{C}$. The map $P \mapsto -P$ is an
involution; the quotient $E/\pm$ is a projective line $\mathbb{P}^1$. The quotient map is 2-to-1 except at
the four points where $P = -P$, i.e. the four points of order dividing 2. This is the classical
"$y^2 = $ quartic $\to x$" picture.

Now go up one dimension. An **abelian surface** $A$ is the 2-dimensional analogue of an elliptic curve: a
projective group variety of dimension 2. The simplest examples are products $E \times E'$ of two elliptic
curves. Again $a \mapsto -a$ is an involution, and again it has fixed points: exactly the 16 points of
$$A[2] = \{a \in A : 2a = 0\} \cong (\mathbb{Z}/2)^4 .$$

The quotient $A/\pm$ is a surface with 16 singular points, one over each point of $A[2]$. Each singularity
is an ordinary double point (a cone), and blowing it up replaces it by a curve. The result
$$\operatorname{Kum}(A) \;=\; \widetilde{A/\pm}$$
is a smooth projective surface — in fact a **K3 surface** — called the *Kummer surface* of $A$. It comes
with 16 distinguished curves $E_1,\dots,E_{16}$, one over each point of $A[2]$. They are pairwise disjoint
(the 16 points are distinct), each is a copy of $\mathbb{P}^1$, and each has self-intersection $-2$.

**The converse (Nikulin, 1975).** This configuration characterises Kummer surfaces. If $Y$ is a K3 surface
containing 16 pairwise disjoint smooth rational curves $E_1,\dots,E_{16}$ whose sum $\sum E_i$ is divisible
by $2$ in $\operatorname{Pic}(Y)$, then $Y \cong \operatorname{Kum}(A)$ for some abelian surface $A$, and the
$E_i$ are exactly the 16 curves above the 2-torsion.

We will call such a set of 16 curves a **Kummer structure** on $Y$. So:

> *"$Y$ is a Kummer surface" $=$ "$Y$ has a Kummer structure".*

This is the crucial move, because a Kummer structure is a purely **combinatorial / lattice-theoretic**
object: 16 divisor classes, with prescribed intersection numbers, whose sum is 2-divisible. That is
something a computer can enumerate.

### Is the abelian surface determined by the Kummer surface?

This deserves saying early, because "*the* Kummer surface of $A$" is a slightly slippery phrase and the
distinction drives everything later.

A single K3 surface can carry **several different Kummer structures** — several different sets of 16
disjoint curves, each of which can be blown down to produce an abelian surface. In general the abelian
surfaces so obtained need **not** be isomorphic: Hosono, Lian, Oguiso and Yau showed that
$\operatorname{Kum}(A) \cong \operatorname{Kum}(B)$ does not force $A \cong B$, and that the number of
non-isomorphic abelian surfaces sharing one Kummer surface can be made arbitrarily large.

For the surface in this notebook, though, geometry pins $A$ down completely. Our $\bar X$ has Picard rank
$20$, the largest possible — a so-called **singular** K3 surface. Such a surface is determined up to
isomorphism by its *transcendental lattice* $T$, the rank-2 positive definite complement of
$\operatorname{Pic}$ inside $H^2$ (Shioda–Inose). Since $T(\operatorname{Kum}(A)) = T(A)(2)$, knowing $T$
determines $T(A)$; and a singular abelian surface is in turn determined up to isomorphism by $T(A)$
(Shioda–Mitani). So over $\bar{\mathbb{Q}}$ **every** Kummer structure on $\bar X$ blows down to the same
abelian surface $\bar A \cong E_i \times E_{2i}$. Section 14 checks a lattice-level shadow of this: all 96
structures used below have isometric orthogonal complements.

What genuinely varies is the **arithmetic**. A Galois-stable Kummer structure is *descent data*: it is the
instruction for how to blow down over the ground field rather than over $\bar{\mathbb{Q}}$. Two different
stable structures, on the same surface over the same field, descend to different objects — different
torsors $V$, under different $\mathbb{Q}$-forms $A$ of the one geometric $\bar A$. That is exactly what
section 14 observes: among the 16 stable structures on $x^4+y^4+4z^4+4w^4 = 0$, some give an $A$ that is a
product of elliptic curves over $\mathbb{Q}$ and others do not, even though all 16 give the same
$\bar A$ over $\bar{\mathbb{Q}}$.

## 2. What changes over a field that is not algebraically closed

Everything above happens over $\bar{\mathbb{Q}}$. Now suppose our surface $Y$ is defined by equations with
coefficients in a field $F \subseteq \bar{\mathbb{Q}}$. The Galois group $\Gamma_F = \operatorname{Gal}(\bar{\mathbb{Q}}/F)$
acts on everything defined over $\bar{\mathbb{Q}}$: on points, on curves, and hence on divisor classes.

A curve on $Y_{\bar{\mathbb{Q}}}$ need not be defined over $F$; Galois moves it around. What we can ask is
that a *finite set* of curves be preserved **as a set**:

> $Y$ is a Kummer surface **over $F$** essentially when $Y$ carries a Kummer structure
> $\{E_1,\dots,E_{16}\}$ that $\Gamma_F$ permutes among themselves.

If that happens, the whole construction descends: the 16 curves can be blown down over $F$, and one obtains
$Y \cong \operatorname{Kum}(V)$ where $V$ is a **torsor** (a "principal homogeneous space", a twisted form
without a marked origin) under an abelian surface $A$ over $F$.

**Torsor versus abelian surface.** There is one further wrinkle, and it will matter at the end. The 16
curves correspond to the 16 points of $A[2]$. If $Y = \operatorname{Kum}(A)$ for an honest abelian surface
$A$ over $F$, then $A$ has a rational origin $0 \in A(F)$, and the curve sitting over $0$ is fixed by every
element of $\Gamma_F$. So:

> If Galois permutes the 16 curves **without fixing any of them**, then $Y$ is $\operatorname{Kum}(V)$ for a
> nontrivial torsor $V$, and $Y$ is *not* $\operatorname{Kum}(A)$ for any abelian surface $A$ over $F$ via
> that structure.

Keep this in mind; every configuration we find below will turn out to be fixed-point free.

## 3. Twisting, and cocycles

Two surfaces over $\mathbb{Q}$ can become isomorphic after enlarging the field. The standard bookkeeping
device for this is a *cocycle*.

Suppose $X$ and $K$ are both defined over $\mathbb{Q}$ and $\psi : \bar X \to \bar K$ is an isomorphism
defined over $k$. Being defined over $\mathbb{Q}$ means each of $\bar X, \bar K$ carries an action of
$\Gamma = \operatorname{Gal}(k/\mathbb{Q})$ (semilinear: it moves points *and* coefficients). Write
$\sigma_X$ and $\sigma_K$ for these. The isomorphism $\psi$ generally fails to commute with them, and the
failure is measured by
$$c_\sigma \;=\; \sigma_K \circ \psi \circ \sigma_X^{-1} \circ \psi^{-1} \;\in\; \operatorname{Aut}(\bar K).$$
One checks $c_{\sigma\tau} = c_\sigma \cdot {}^\sigma c_\tau$: a **1-cocycle**. And $\psi$ descends to
$\mathbb{Q}$ exactly when $c$ is trivial. Since $\psi$ was only ever a choice, what matters is $c$ up to
*coboundary*: replacing $\psi$ by $\alpha \circ \psi$ replaces $c_\sigma$ by
$\alpha\, c_\sigma\, {}^\sigma\alpha^{-1}$.

**Transporting to one side.** Set $\gamma_\sigma = \psi^{-1} c_\sigma \psi \in \operatorname{Aut}(\bar X)$.
Now twist: let $a$ be a cocycle valued in $\operatorname{Aut}(\bar X)$, giving a twisted form $X'$ with
Galois action $a_\sigma \sigma_X$, and let $b$ be one valued in $\operatorname{Aut}(\bar K)$, giving $K'$.
A short computation gives the twisted actions $a_\sigma\sigma_X$ and $b^\psi_\sigma \gamma_\sigma \sigma_X$
on the *same* space $\bar X$, so:

> $$X' \cong K' \ \text{over } k' \quad\Longleftrightarrow\quad a \ \text{and}\ b^\psi\!\cdot\gamma
>   \ \text{are cohomologous in } H^1(\Gamma_{k'}, \operatorname{Aut}\bar X).$$

**The two constraints in our problem.** "$X'$ is still a diagonal quartic" says $a$ takes values in the
group $D = \mu_4^4/\mu_4$ of diagonal automorphisms. "$K'$ is still a Kummer surface" says $b$ preserves a
Kummer structure, i.e. $b$ takes values in the stabiliser $S$ of one.

**The reformulation we actually use.** Let $\mathcal{N}$ be the (finite) set of all Kummer structures on
$\bar X$. The group $\operatorname{Aut}(\bar X)$ acts on it, with $S$ the stabiliser of a chosen one. The
displayed criterion then becomes a **fixed-point problem**:

> $X'$ is isomorphic over $k'$ to some Kummer surface $\iff$ the twisted $\Gamma_{k'}$-action on
> $\mathcal{N}$ has a fixed point.

This is exactly the classical statement "a twisted form has a rational point iff its class comes from the
stabiliser", and it is what makes the whole question finite. Note what it does *not* do: it does not give a
closed formula. $\operatorname{Aut}(\bar X)$ is an infinite group here, and the image of
$H^1(k',S) \to H^1(k',\operatorname{Aut}\bar X)$ has no simple description. But counting fixed points on a
finite set is a computation we can actually run.

## 4. Why this is linear algebra

To compute we need a concrete model of "divisor classes with their intersection numbers". That is the
**Picard lattice** $\operatorname{Pic}(\bar X)$: the group of divisor classes, equipped with the
intersection pairing $D \cdot D'$, a symmetric bilinear form with integer values.

For our $X$ this lattice is completely understood. Pinch and Swinnerton-Dyer proved that
$\operatorname{Pic}(\bar X)$ is generated by the **48 lines** lying on the Fermat quartic, and has rank 20
(the largest possible for a K3 surface). Concretely the lines are, for $\mu,\nu$ odd,
$$L_{\mu\nu}: \; x_0 = \epsilon^\mu x_1,\ x_2 = \epsilon^\nu x_3, \qquad
  M_{\mu\nu}: \; x_0 = \epsilon^\mu x_2,\ x_1 = \epsilon^\nu x_3, \qquad
  N_{\mu\nu}: \; x_0 = \epsilon^\mu x_3,\ x_1 = \epsilon^\nu x_2,$$
where $\epsilon = \zeta_8$; that is $3 \times 4 \times 4 = 48$ lines, all defined over $k = \mathbb{Q}(\mu_8)$.

Two facts make the computation possible:

- Every line has self-intersection $-2$ and every curve class we need is an integer combination of lines.
- Galois **permutes the 48 lines**, so the Galois action on the whole lattice is determined by a permutation
  of 48 objects. Same for the diagonal automorphisms.

So: the Galois action becomes a $20\times20$ integer matrix, a Kummer structure becomes a set of 16 integer
vectors, and "is there a stable Kummer structure" becomes a finite check.

Let us build all of this.

## 5. The 48 lines and their intersection numbers

Each line is given as the row space of a $2\times4$ matrix over $\mathbb{Q}(\mu_8)$ — that is, as the set of
points $s\cdot(\text{row}_1) + t\cdot(\text{row}_2)$. Two distinct lines in $\mathbb{P}^3$ meet iff the
$4\times4$ matrix stacking their two bases is singular; on a smooth surface two distinct curves meeting
transversally in one point have intersection number 1.

In [1]:
K.<ep> = CyclotomicField(8)          # ep = zeta_8, a primitive 8th root of unity
odds = [1,3,5,7]

# family L: x0 = ep^mu x1, x2 = ep^nu x3   (and cyclic variants M, N)
spec = {'L': (0,1,2,3), 'M': (0,2,1,3), 'N': (0,3,1,2)}
lines = []
for fam in ['L','M','N']:
    c = spec[fam]
    for mu in odds:
        for nu in odds:
            B = matrix(K, 2, 4)
            B[0,c[0]] = ep^mu ; B[0,c[1]] = 1
            B[1,c[2]] = ep^nu ; B[1,c[3]] = 1
            lines.append((fam, mu, nu, B))
n = len(lines)
print("number of lines:", n)

# every one of them really lies on the Fermat quartic
R.<s,t> = PolynomialRing(K)
bad = 0
for (f,mu,nu,B) in lines:
    pt = [s*B[0][j] + t*B[1][j] for j in range(4)]
    if sum(u^4 for u in pt) != 0: bad += 1
print("lines failing to lie on x0^4+x1^4+x2^4+x3^4 = 0 :", bad)

number of lines: 48
lines failing to lie on x0^4+x1^4+x2^4+x3^4 = 0 : 0


In [2]:
# the 48 x 48 matrix of intersection numbers
Gram = matrix(ZZ, n, n)
for a in range(n):
    for b in range(n):
        if a == b:
            Gram[a,b] = -2                                   # a line on a K3 has self-intersection -2
        else:
            Gram[a,b] = 1 if lines[a][3].stack(lines[b][3]).rank() < 4 else 0
labels = [(f,mu,nu) for (f,mu,nu,B) in lines]
idx = {labels[a]: a for a in range(n)}
print("rank of the lattice spanned by the 48 lines :", Gram.rank())
print("(20 is the maximum possible for a K3 surface, and is what Pinch-Swinnerton-Dyer prove)")

rank of the lattice spanned by the 48 lines : 20
(20 is the maximum possible for a K3 surface, and is what Pinch-Swinnerton-Dyer prove)


## 6. Coordinates on the Picard lattice

Pick 20 of the lines forming a basis of $\operatorname{Pic}\otimes\mathbb{Q}$. Because the intersection form
is nondegenerate, any divisor class is determined by its intersection numbers with those 20 lines, and we
use those numbers as coordinates. Two useful checks come for free: the discriminant of the lattice should be
$-64$, and the hyperplane class $H$ (a plane section, self-intersection 4) should meet every line once.

In [3]:
bas = []
for a in range(n):
    if Gram[bas+[a], bas+[a]].rank() == len(bas)+1: bas.append(a)
GB = Gram[bas, bas]
GBinv = GB.inverse()
# coordinates of a class = GB^{-1} * (its intersection numbers with the 20 basis lines)
linecoord = [GBinv * vector(QQ, [Gram[a,b] for b in bas]) for a in range(n)]
def pair(u,v): return u * GB * v                       # the intersection pairing in these coordinates

assert all(pair(linecoord[a], linecoord[b]) == Gram[a,b] for a in range(n) for b in range(n))
print("pairing reproduces all 48 x 48 intersection numbers : OK")

Pic  = matrix(QQ, linecoord).row_module(ZZ)            # Pic = Z-span of the 48 line classes
Bmat = matrix(QQ, Pic.basis())
print("rank of Pic :", Pic.rank(), "   discriminant :", (Bmat*GB*Bmat.transpose()).det())

H = sum(linecoord[idx[('L',1,nu)]] for nu in odds)     # a plane section
Hv = vector(QQ, H)
print("H^2 =", pair(Hv,Hv), "  and H meets every line in", sorted(set(pair(Hv,linecoord[a]) for a in range(n))), "point")

pairing reproduces all 48 x 48 intersection numbers : OK
rank of Pic : 20    discriminant : -64
H^2 = 4   and H meets every line in [1] point


## 7. How Galois acts

$\Gamma = \operatorname{Gal}(\mathbb{Q}(\mu_8)/\mathbb{Q}) = \{1,\sigma_3,\sigma_5,\sigma_7\} \cong (\mathbb{Z}/2)^2$,
where $\sigma_m(\epsilon) = \epsilon^m$. Applying $\sigma_m$ to the equations of $L_{\mu\nu}$ turns them into
those of $L_{m\mu,\,m\nu}$, so Galois simply multiplies the indices:
$$\sigma_m : \ (f,\mu,\nu) \longmapsto (f,\ m\mu \bmod 8,\ m\nu \bmod 8).$$
The three subgroups of order 2 correspond to the three quadratic subfields — the fixed field of $\sigma_m$
is the one $\sigma_m$ does not move:

| generator | fixes |
|---|---|
| $\sigma_3$ | $\mathbb{Q}(\sqrt{-2})$ |
| $\sigma_5$ | $\mathbb{Q}(i)$ |
| $\sigma_7$ | $\mathbb{Q}(\sqrt{2})$ |

In [4]:
def p_sigma(m): return [idx[(f,(m*mu)%8,(m*nu)%8)] for (f,mu,nu) in labels]

def act(perm):
    "the 20x20 matrix of the isometry of Pic induced by a permutation of the 48 lines"
    return matrix(QQ, [linecoord[perm[b]] for b in bas])

I20 = identity_matrix(QQ,20)
sub = {3:"Q(sqrt-2)", 5:"Q(i)", 7:"Q(sqrt2)"}
Gact = {}
for m in (3,5,7):
    p = p_sigma(m); M = act(p); Gact[m] = M
    iso = all(pair(vector(QQ,linecoord[a])*M, vector(QQ,linecoord[b])*M) == Gram[a,b]
              for a in range(n) for b in range(n))
    print("sigma_%d (fixes %-10s): isometry %s, fixes H %s, involution %s, rank Pic^sigma = %2d"
          % (m, sub[m], iso, Hv*M == Hv, M*M == I20, (M-I20).right_kernel().dimension()))
rQ = matrix(QQ, (Gact[3]-I20).rows() + (Gact[5]-I20).rows()).right_kernel().dimension()
print("full group Gal(Q(mu_8)/Q)                  : rank Pic^Gamma = %2d" % rQ)

sigma_3 (fixes Q(sqrt-2) ): isometry True, fixes H True, involution True, rank Pic^sigma = 11


sigma_5 (fixes Q(i)      ): isometry True, fixes H True, involution True, rank Pic^sigma =  8


sigma_7 (fixes Q(sqrt2)  ): isometry True, fixes H True, involution True, rank Pic^sigma = 11
full group Gal(Q(mu_8)/Q)                  : rank Pic^Gamma =  5


## 8. The diagonal twists, and the twisted Galois action

The twisted surface
$$X_{a,b,c,d}: \; a x_0^4 + b x_1^4 + c x_2^4 + d x_3^4 = 0$$
becomes the Fermat quartic over $\bar{\mathbb{Q}}$ via $\phi:\ x_i \mapsto a_i^{1/4} x_i$ (writing
$a_0,\dots,a_3$ for $a,b,c,d$). Transporting the Galois action of $X_{a,b,c,d}$ along $\phi$ gives, on the
Fermat quartic, the *twisted* action
$$\sigma \;\longmapsto\; \delta_\sigma \circ \sigma, \qquad
  \delta_\sigma = \operatorname{diag}\bigl(\sigma(a_i^{1/4})/a_i^{1/4}\bigr)_i \in D = \mu_4^4/\mu_4 .$$

We restrict to the twists for which the fourth roots already lie in $k$, so that the 48 lines stay defined
over $\mathbb{Q}(\mu_8)$ and $\Gamma$ stays of order 4. That means all $a_i$ lie in the subgroup
$\langle -1, 4\rangle \subset \mathbb{Q}^*/\mathbb{Q}^{*4}$ (note $4^2 = 2^4$ is a fourth power, so this
group has order 4) — exactly the group $H_D$ of Ieronymou–Skorobogatov–Zarhin. Normalising $a=1$ leaves
$4^3 = 64$ twists.

Two things are checked below before anything is built on them: that the recorded ratios
$\sigma_m(a_i^{1/4})/a_i^{1/4}$ are correct, and that the resulting twisted permutations really satisfy
the multiplication table of $(\mathbb{Z}/2)^2$ — i.e. that they define a genuine Galois action and not just
three unrelated maps.

In [5]:
# a chosen fourth root of each allowed twist value
root = {1: K(1), -1: ep, 4: ep+ep^-1, -4: 1+ep^2}      # note (ep+ep^-1)^4 = 4, (1+i)^4 = -4
Etab = {}
for v,u in root.items():
    assert u^4 == v
    Etab[v] = {1: 0}
    for m in (3,5,7):
        r = K.hom([ep^m])(u)/u                          # sigma_m(u)/u, a fourth root of unity
        e = [j for j in range(4) if r == ep^(2*j)][0]
        Etab[v][m] = e
print("Etab[value][m] = e  with  sigma_m(u)/u = ep^(2e) :")
for v in (1,-1,4,-4): print("   value %3d : %s" % (v, Etab[v]))

Etab[value][m] = e  with  sigma_m(u)/u = ep^(2e) :
   value   1 : {1: 0, 3: 0, 5: 0, 7: 0}
   value  -1 : {1: 0, 3: 1, 5: 2, 7: 3}
   value   4 : {1: 0, 3: 2, 5: 2, 7: 0}
   value  -4 : {1: 0, 3: 3, 5: 0, 7: 3}


In [6]:
def p_delta(e):
    "the diagonal automorphism diag(ep^(2e_0),...,ep^(2e_3)), as a permutation of the 48 lines"
    out = []
    for (f,mu,nu) in labels:
        if   f=='L': s,t = 2*(e[0]-e[1]), 2*(e[2]-e[3])
        elif f=='M': s,t = 2*(e[0]-e[2]), 2*(e[1]-e[3])
        else:        s,t = 2*(e[0]-e[3]), 2*(e[1]-e[2])
        out.append(idx[(f,(mu+s)%8,(nu+t)%8)])
    return out

def compose(p,q): return [p[q[i]] for i in range(n)]
def tperm(A,m):   return compose(p_delta([Etab[A[i]][m] for i in range(4)]), p_sigma(m))

import itertools
twists = list(itertools.product([1],[1,-1,4,-4],[1,-1,4,-4],[1,-1,4,-4]))
idperm = list(range(n))
bad = 0
for A in twists:
    t = {m: tperm(A,m) for m in (3,5,7)}
    if compose(t[3],t[5]) != t[7] or compose(t[5],t[3]) != t[7]: bad += 1
    if any(compose(t[m],t[m]) != idperm for m in (3,5,7)): bad += 1
    if any(Gram[t[m][a],t[m][b]] != Gram[a,b] for m in (3,) for a in range(n) for b in range(0,n,7)): bad += 1
print("twists (of %d) failing the group law, involutivity, or preservation of the form : %d" % (len(twists), bad))

twists (of 64) failing the group law, involutivity, or preservation of the form : 0


## 9. A test that costs nothing: counting ranks

Before searching for anything, there is a numerical obstruction that rules out cases outright.

Suppose $Y$ over $F$ has a Galois-stable Kummer structure $\Omega = \{E_1,\dots,E_{16}\}$. The 16 classes are
independent, and their orthogonal complement inside $\operatorname{Pic}(\bar Y)$ has rank $20 - 16 = 4$ and
is (up to scaling the form) the Néron–Severi group of the abelian surface. So as $\Gamma_F$-modules
$$\operatorname{Pic}(\bar Y)\otimes\mathbb{Q} \;\cong\; \mathbb{Q}[\Omega] \;\oplus\; \operatorname{NS}(\bar A)\otimes\mathbb{Q}.$$
Taking invariants — and using that the invariants of a permutation module $\mathbb{Q}[\Omega]$ have dimension
equal to the **number of orbits** — gives the identity
$$\boxed{\ \operatorname{rank}\operatorname{Pic}(\bar Y)^{\Gamma_F} \;=\; \#(\Omega/\Gamma_F)\;+\;\operatorname{rank}\operatorname{NS}(\bar A)^{\Gamma_F}. \ }$$
The last term is at least 1, because a projective variety over $F$ has an ample class defined over $F$.

Now specialise:

- $F = k'$ **quadratic**, so $\operatorname{Gal}(k/k') = \mathbb{Z}/2$. Orbits have size 1 or 2, so there are
  at least 8 of them, and therefore $\operatorname{rank}\operatorname{Pic}^{\sigma} \ge 9$.
- $F = \mathbb{Q}$, so $\operatorname{Gal}(k/\mathbb{Q}) = (\mathbb{Z}/2)^2$. Orbits have size at most 4, so
  there are at least 4 of them, and $\operatorname{rank}\operatorname{Pic}^{\Gamma} \ge 5$.

For the Fermat quartic the three quadratic ranks came out as $11, 8, 11$ in section 7. The middle one is
below 9, so:

> **$\mathbb{Q}(i)$ is excluded immediately** — the Fermat quartic is not a Kummer surface over
> $\mathbb{Q}(i)$, and no search is required to see it.

Here is the same test applied to all 64 twists and all three quadratic subfields.

In [7]:
survive = {3: [], 5: [], 7: []}
ranks = {}
for A in twists:
    for m in (3,5,7):
        M = act(tperm(A,m))
        r = (M-I20).right_kernel().dimension()
        ranks[(A,m)] = r
        if r >= 9: survive[m].append(A)
print("rank Pic^sigma over each quadratic subfield (need >= 9):\n")
print("   twist             Q(sqrt-2)  Q(i)  Q(sqrt2)")
for A in twists[:8]:
    print("   %-16s   %2d      %2d      %2d" % (str(A), ranks[(A,3)], ranks[(A,5)], ranks[(A,7)]))
print("   ... (%d twists in total)\n" % len(twists))
for m in (3,5,7):
    print("%-10s : %2d of 64 twists pass the rank test" % (sub[m], len(survive[m])))
print()
print("distinct rank values observed :", sorted(set(ranks.values())))

rank Pic^sigma over each quadratic subfield (need >= 9):

   twist             Q(sqrt-2)  Q(i)  Q(sqrt2)
   (1, 1, 1, 1)       11       8      11
   (1, 1, 1, -1)      10       8      10
   (1, 1, 1, 4)       11       8      11
   (1, 1, 1, -4)      10       8      10
   (1, 1, -1, 1)      10       8      10
   (1, 1, -1, -1)     11      16      11
   (1, 1, -1, 4)      10      16      10
   (1, 1, -1, -4)     11       8      11
   ... (64 twists in total)

Q(sqrt-2)  : 64 of 64 twists pass the rank test
Q(i)       : 24 of 64 twists pass the rank test
Q(sqrt2)   : 64 of 64 twists pass the rank test

distinct rank values observed : [8, 10, 11, 16]


## 10. A supply of Kummer structures

Now we need actual Kummer structures on $\bar X$ to test. Swinnerton-Dyer's appendix exhibits one: eight of
the 48 lines, together with eight **conics**. A conic here is the residual curve when a plane through two
meeting lines $\Lambda, \Lambda'$ is intersected with the quartic surface: the plane section is a quartic
curve containing the two lines, and what is left over is a conic, of class
$$[\Lambda\Lambda'] \;=\; H - \Lambda - \Lambda'.$$

To get *many* Kummer structures cheaply, we move this one around by symmetries. Any permutation of the 48
lines preserving all their intersection numbers extends (uniquely, since the form is nondegenerate) to an
isometry of $\operatorname{Pic}$; such permutations are exactly the automorphisms of the *intersection graph*
of the 48 lines. If in addition the isometry fixes $H$, it sends lines to lines and conic classes to conic
classes, hence sends genuine curves to genuine curves — so it carries our Kummer structure to another one.

In [8]:
Gr = Graph(n)
Gr.add_edges([(a,b) for a in range(n) for b in range(a+1,n) if Gram[a,b]==1])
AG = Gr.automorphism_group()
print("automorphisms of the 48-line intersection graph :", AG.order())
print("all generators fix H :", all(Hv*act([g(a) for a in range(n)]) == Hv for g in AG.gens()))

automorphisms of the 48-line intersection graph : 6144
all generators fix H : True


In [9]:
def vec_line(a):    return vector(QQ, linecoord[a])
def vec_conic(a,b): return Hv - vec_line(a) - vec_line(b)

# Swinnerton-Dyer's configuration: 8 lines + 8 conics
N0_lines  = [('M',5,1),('M',3,3),('M',1,5),('M',7,7),('N',1,1),('N',3,7),('N',5,5),('N',7,3)]
N0_conics = [(('L',3,3),('M',1,1)), (('L',3,3),('M',5,5)), (('L',1,5),('M',3,7)), (('L',1,5),('M',7,3)),
             (('L',5,7),('N',1,5)), (('L',5,7),('N',5,1)), (('L',7,1),('N',3,3)), (('L',7,1),('N',7,7))]
N0 = (frozenset(idx[x] for x in N0_lines),
      frozenset(tuple(sorted((idx[u], idx[v]))) for (u,v) in N0_conics))

def vecs(cfg):
    L, C = cfg
    return [vec_line(a) for a in L] + [vec_conic(a,b) for (a,b) in C]

V = vecs(N0)
print("Swinnerton-Dyer's 16 classes:")
print("   all self-intersection -2 :", all(pair(v,v) == -2 for v in V))
print("   pairwise disjoint        :", all(pair(V[i],V[j]) == 0 for i in range(16) for j in range(i+1,16)))
print("   half of the sum is in Pic:", (sum(V)/2) in Pic)

Swinnerton-Dyer's 16 classes:
   all self-intersection -2 : True
   pairwise disjoint        : True
   half of the sum is in Pic: True


In [10]:
# move it around by the 6144 graph automorphisms
def apply_perm(p, cfg):
    L, C = cfg
    return (frozenset(p[a] for a in L),
            frozenset(tuple(sorted((p[a], p[b]))) for (a,b) in C))

orbit = set()
for g in AG:
    orbit.add(apply_perm([g(a) for a in range(n)], N0))
orbit = sorted(orbit, key=lambda c: (sorted(c[0]), sorted(c[1])))
print("distinct Kummer structures obtained :", len(orbit))

# and check every single one really is a Kummer structure made of irreducible curves
lineset = set(tuple(vec_line(a)) for a in range(n))
def is_kummer_structure(cfg):
    V = vecs(cfg)
    if len(V) != 16: return False
    if not all(pair(v,v) == -2 for v in V): return False
    if not all(pair(V[i],V[j]) == 0 for i in range(16) for j in range(i+1,16)): return False
    if (sum(V)/2) not in Pic: return False
    # a degree-2 class is reducible exactly when it is a sum of two lines
    for v in V:
        if pair(v,Hv) == 2 and any(tuple(v - vec_line(a)) in lineset for a in range(n)): return False
    return True
print("of these, verified genuine (disjoint, 2-divisible, irreducible) :",
      sum(1 for c in orbit if is_kummer_structure(c)))
print("degrees occurring in each :", sorted(set(pair(v,Hv) for c in orbit for v in vecs(c))))

distinct Kummer structures obtained : 96


of these, verified genuine (disjoint, 2-divisible, irreducible) : 96


degrees occurring in each : [1, 2]


## 11. The search

Everything is now in place. For each of the 64 twists and each subfield $k'$, we ask whether any of these
96 Kummer structures is preserved as a set by the twisted Galois action.

Because a Kummer structure is recorded combinatorially (which lines, which pairs of lines), applying a
permutation is just relabelling, and the whole sweep takes seconds.

In [11]:
orbit_set = set(orbit)
def stable_count(A, ms):
    "how many of the 96 structures are preserved by the twisted action of the sigma_m, m in ms"
    perms = [tperm(A,m) for m in ms]
    return sum(1 for c in orbit if all(apply_perm(p,c) == c for p in perms))

print("number of the 96 Kummer structures preserved by the twisted Galois action")
print("('.' means none)\n")
print("   twist             Q(sqrt-2)  Q(i)  Q(sqrt2)    Q")
tot = {}
for A in twists:
    row = [stable_count(A,(m,)) for m in (3,5,7)] + [stable_count(A,(3,5,7))]
    tot[A] = row
    if any(row):
        f = lambda c: ("%3d" % c) if c else "  ."
        print("   %-16s  %s     %s    %s     %s" % (str(A), f(row[0]), f(row[1]), f(row[2]), f(row[3])))
print()
for j,name in enumerate(["Q(sqrt-2)","Q(i)","Q(sqrt2)","Q"]):
    hit = [A for A in twists if tot[A][j]]
    print("%-10s : %2d of 64 twists carry a stable Kummer structure" % (name, len(hit)))

number of the 96 Kummer structures preserved by the twisted Galois action
('.' means none)

   twist             Q(sqrt-2)  Q(i)  Q(sqrt2)    Q
   (1, 1, 1, 1)       24       .     24       .
   (1, 1, 1, 4)       24       .     24       .
   (1, 1, -1, -1)      .      64      .       .
   (1, 1, -1, 4)       .      64      .       .
   (1, 1, 4, 1)       24       .     24       .
   (1, 1, 4, -1)       .      64      .       .
   (1, 1, 4, 4)       24      64     24      16
   (1, -1, 1, -1)      .      64      .       .
   (1, -1, 1, 4)       .      64      .       .
   (1, -1, -1, 1)      .      64      .       .
   (1, -1, -1, -4)     .      64      .       .
   (1, -1, 4, 1)       .      64      .       .
   (1, -1, 4, -4)      .      64      .       .
   (1, -1, -4, -1)     .      64      .       .
   (1, -1, -4, 4)      .      64      .       .
   (1, 4, 1, 1)       24       .     24       .
   (1, 4, 1, -1)       .      64      .       .
   (1, 4, 1, 4)       24      64     24 

Read off the two headline facts.

**The Fermat quartic itself**, twist $(1,1,1,1)$: 24 stable structures over $\mathbb{Q}(\sqrt{-2})$, 24 over
$\mathbb{Q}(\sqrt2)$, none over $\mathbb{Q}(i)$ (as the rank count already told us) and none over
$\mathbb{Q}$. So Mizukami's isomorphism does descend — not to $\mathbb{Q}$, but from a degree-4 field to a
degree-2 one, and with no twisting at all.

**Over $\mathbb{Q}$**, three twists succeed, and they are one surface up to permuting the coordinates:
$$x^4 + y^4 + 4z^4 + 4w^4 = 0 .$$

Let us look at an actual configuration in each case.

In [12]:
def show(A, ms, tag):
    perms = [tperm(A,m) for m in ms]
    good = [c for c in orbit if all(apply_perm(p,c) == c for p in perms)]
    print("=== %s : %d stable structures ; here is one ===" % (tag, len(good)))
    L, C = good[0]
    for a in sorted(L):    print("    line  %s%d%d" % labels[a])
    for (a,b) in sorted(C): print("    conic [%s%d%d %s%d%d]" % (labels[a] + labels[b]))
    V = vecs(good[0])
    print("    checks: (-2) %s, disjoint %s, 2-divisible %s, irreducible %s"
          % (all(pair(v,v)==-2 for v in V),
             all(pair(V[i],V[j])==0 for i in range(16) for j in range(i+1,16)),
             (sum(V)/2) in Pic, is_kummer_structure(good[0])))
    print()

show((1,1,1,1), (3,),    "Fermat quartic over Q(sqrt-2)")
show((1,1,1,1), (7,),    "Fermat quartic over Q(sqrt2)")
show((1,1,4,4), (3,5,7), "x^4+y^4+4z^4+4w^4 = 0 over Q")

=== Fermat quartic over Q(sqrt-2) : 24 stable structures ; here is one ===
    line  L11
    line  L33
    line  L55
    line  L77
    line  M15
    line  M37
    line  M51
    line  M73
    conic [L15 N13]
    conic [L37 N31]
    conic [L51 N13]
    conic [L73 N31]
    conic [M11 N55]
    conic [M33 N77]
    conic [M55 N55]
    conic [M77 N77]
    checks: (-2) True, disjoint True, 2-divisible True, irreducible True

=== Fermat quartic over Q(sqrt2) : 24 stable structures ; here is one ===
    line  L11
    line  L33
    line  L55
    line  L77
    line  M15
    line  M37
    line  M51
    line  M73
    conic [L15 N13]
    conic [L37 N75]
    conic [L51 N13]
    conic [L73 N75]
    conic [M11 N37]
    conic [M33 N51]
    conic [M55 N37]
    conic [M77 N51]
    checks: (-2) True, disjoint True, 2-divisible True, irreducible True

=== x^4+y^4+4z^4+4w^4 = 0 over Q : 16 stable structures ; here is one ===
    line  L11
    line  L33
    line  L55
    line  L77
    line  M15
    line  M37
 

    checks: (-2) True, disjoint True, 2-divisible True, irreducible True



Notice that the configuration printed for $x^4+y^4+4z^4+4w^4=0$ over $\mathbb{Q}$ is *the same one* printed
for the Fermat quartic over $\mathbb{Q}(\sqrt2)$. That is not a coincidence, and it explains why this
particular twist is the one that works.

Look back at the table of $\sigma_m(u)/u$ in section 8: for the twist values $1$ and $4$ the entry at
$m=7$ is $0$. So the twist $(1,1,4,4)$ changes nothing at all at $\sigma_7$ — the twisted action of
$\sigma_7$ *is* the untwisted one. The Fermat quartic was already a Kummer surface over
$\mathbb{Q}(\sqrt2)$, i.e. already stable under $\sigma_7$; what the twist does is repair the remaining
two elements $\sigma_3, \sigma_5$ without disturbing $\sigma_7$. The 16 structures stable over
$\mathbb{Q}$ are exactly those 16 of the 24 $\sigma_7$-stable ones that survive the other two.

## 12. An independent check

The method above only ever looks at 96 particular configurations, so a positive answer is airtight but it is
worth confirming by a completely different route. Here is one: forget the graph symmetries and instead
**enumerate all $(-2)$-classes** of small degree that are orthogonal to their own Galois conjugates, group
them into orbits, and search for 16 of them that are pairwise disjoint with 2-divisible sum.

This is the brute-force method, done by short-vector enumeration in the invariant sublattice. It is slower,
but it is independent — and it is also where a subtle trap lies, so the code below is written carefully.

**The trap.** After enumerating *classes* one must discard those that are not irreducible *curves*. The test
is: if $D$ is effective and $C$ is an irreducible curve with $D \cdot C < 0$, then $C$ is a component of $D$.
So an effective class $E$ of small degree with $E^2 = -2$ is reducible as soon as $E \cdot L < 0$ for some
line $L$ — **but one must exclude $L = E$ itself**, since every line satisfies $E\cdot E = -2 < 0$ and is
nevertheless perfectly irreducible. Omitting that exclusion silently discards all 48 lines, and with them
every configuration containing a line.

In [13]:
Bm = matrix(QQ, Pic.basis())                 # write everything in a Z-basis of Pic
GP = Bm*GB*Bm.transpose()
def toPic(v): return Bm.solve_left(matrix(QQ,[v]))[0]
def ip(u,v):  return u*GP*v
hG = vector(QQ, toPic(Hv))
lc = [vector(ZZ, toPic(vec_line(a))) for a in range(n)]
E20 = identity_matrix(QQ,20)
print("the 48 lines have degree", sorted(set(ip(L,hG) for L in lc)),
      "and self-intersection", sorted(set(ip(L,L) for L in lc)))

def act_matrix(perm):
    rows = []
    for e in E20.rows():
        amb = vector(QQ,e)*Bm
        rows.append(vector(QQ, toPic(sum(amb[i]*vec_line(perm[bas[i]]) for i in range(20)))))
    return matrix(ZZ, rows)

def shorts(Bl, targ):
    "all vectors of given square in a negative-definite sublattice, via PARI's qfminim"
    if targ == 0: return [vector(QQ,[0]*20)]
    Grl = Bl*GP*Bl.transpose()
    sc = lcm([QQ(x).denominator() for x in (-2*Grl).list()]+[1])
    Q = matrix(ZZ, sc*(-2*Grl)); tg = sc*(-2*targ); out = []
    for r in pari(Q).qfminim(tg, 10^7, 2)[2].sage().transpose().rows():
        x = vector(ZZ,r)
        if x*Q*x == tg:
            for y in (x,-x): out.append(vector(QQ,y)*Bl)
    return out

the 48 lines have degree [1] and self-intersection [-2]


In [14]:
import time
pari.allocatemem(2*10^9, silent=True)
t0 = time.time()
A = (1,1,4,4); Dbound = 3
M = {m: act_matrix(tperm(A,m)) for m in (3,5,7)}; M[1] = identity_matrix(ZZ,20)
Av = (matrix(QQ,M[1])+M[3]+M[5]+M[7])/4                      # projector onto the invariant part
BG = matrix(QQ, matrix(QQ,Av).row_module(ZZ).basis())        # invariant sublattice
BW = matrix(QQ, matrix(QQ,E20-Av).row_module(ZZ).basis())    # its complement (negative definite)
rg = BG.nrows()
GrG = BG*GP*BG.transpose()
cG  = vector(QQ,[ip(vector(QQ,BG.row(i)), hG) for i in range(rg)])
QG0 = -2*GrG + matrix(QQ,[[cG[i]*cG[j] for j in range(rg)] for i in range(rg)])   # positive definite
sG  = lcm([QQ(x).denominator() for x in QG0.list()]+[1]); QG = matrix(ZZ, sG*QG0)

cand = set()
for s in (1,2,4):                       # s = size of the stabiliser of the class in Gamma
    Xis = shorts(BW, QQ(-2)+QQ(s)/2); e1t = QQ(-s)/2
    for r in pari(QG).qfminim(sG*(-2*e1t + Dbound^2), 10^7, 2)[2].sage().transpose().rows():
        for y in (vector(ZZ,r), -vector(ZZ,r)):
            v = vector(QQ,y)*BG
            if ip(v,v) != e1t or not (1 <= ip(v,hG) <= Dbound): continue
            for x in Xis:
                E = v + x
                if all(c in ZZ for c in E): cand.add(tuple(ZZ(c) for c in E))
allC = [vector(ZZ,c) for c in cand]
print("twist %s : rank Pic^Gamma = %d ; (-2)-classes of degree <= %d orthogonal to their conjugates : %d"
      % (str(A), rg, Dbound, len(allC)))

twist (1, 1, 4, 4) : rank Pic^Gamma = 9 ; (-2)-classes of degree <= 3 orthogonal to their conjugates : 1664


In [15]:
from collections import Counter
def irreducible_correct(E): return all(ip(E,L) >= 0 for L in lc if tuple(L) != tuple(E))
def irreducible_buggy(E):   return all(ip(E,L) >= 0 for L in lc)                 # forgets L = E

def configurations(C):
    if not C: return 0, 0
    key = {tuple(c):i for i,c in enumerate(C)}
    seen = set(); orbs = []
    for i,c in enumerate(C):
        if i in seen: continue
        img = [key.get(tuple(c*M[m])) for m in (1,3,5,7)]
        if any(t is None for t in img): continue
        o = sorted(set(img)); seen |= set(o)
        if all(ip(C[a],C[b]) == 0 for a in o for b in o if a < b): orbs.append(o)
    m = len(orbs)
    if m == 0: return 0, 0
    R = matrix(ZZ,[C[o[0]] for o in orbs])
    PR = {mm: R*GP*(R*M[mm]).transpose() for mm in (1,3,5,7)}
    comp = [[all(PR[mm][i,j] == 0 for mm in (1,3,5,7)) for j in range(m)] for i in range(m)]
    sols = []
    def rec(start, chosen, totl):
        if len(chosen) > rg-1: return
        if totl == 16:
            Ssum = sum(C[t] for o in chosen for t in orbs[o])
            if all(x % 2 == 0 for x in Ssum): sols.append(list(chosen))
            return
        if totl > 16 or len(chosen) == rg-1: return
        for i in range(start, m):
            if len(orbs[i]) + totl > 16: continue
            if all(comp[i][j] for j in chosen):
                chosen.append(i); rec(i+1, chosen, totl+len(orbs[i])); chosen.pop()
    rec(0, [], 0)
    return m, len(sols)

for tag, filt in (("WITHOUT the exclusion L = E (wrong)", irreducible_buggy),
                  ("WITH    the exclusion L = E (right)", irreducible_correct)):
    C = [E for E in allC if filt(E)]
    m, s = configurations(C)
    print("%s : %4d classes kept %-28s -> %2d orbits, %2d configurations"
          % (tag, len(C), str(dict(Counter(ip(E,hG) for E in C))), m, s))
print("\n(the difference is exactly the 48 lines)")
print("elapsed %.0fs" % (time.time()-t0))

WITHOUT the exclusion L = E (wrong) :  192 classes kept {3: 64, 2: 128}              -> 72 orbits,  0 configurations


WITH    the exclusion L = E (right) :  240 classes kept {3: 64, 2: 128, 1: 48}       -> 92 orbits, 16 configurations

(the difference is exactly the 48 lines)
elapsed 37s


The independent enumeration returns **16** configurations for $x^4+y^4+4z^4+4w^4=0$ — the same number the
symmetry method found in section 11, with the same degree profile (eight lines and eight conics). The two
methods agree.

It also shows what the faulty filter does: it keeps 192 classes instead of 240, the 48 missing ones being
precisely the lines, and the count of configurations collapses from 16 to 0. This is worth flagging because
that exact error produced a false negative in an earlier draft of this investigation.

## 13. Abelian surface, or torsor?

One question from section 2 remains: is the object on the other side an abelian surface, or only a torsor?
The test is whether Galois **fixes** one of the 16 curves — the curve over the origin $0 \in A[2]$ is fixed
whenever there is an honest origin to be had.

We also take the opportunity to check the rank identity of section 9 numerically. It should hold on the
nose in every case, and it is a genuinely independent confirmation that these configurations are what they
claim to be.

In [16]:
def analyse(A, ms, tag):
    Ms = [act(tperm(A,m)) for m in ms]
    grp = [I20]
    for M in Ms: grp = grp + [g*M for g in grp]
    rk = matrix(QQ, sum([list((M-I20).rows()) for M in Ms], [])).right_kernel().dimension()
    good = [c for c in orbit if all(apply_perm(tperm(A,m), c) == c for m in ms)]
    if not good:
        print("%-38s rank Pic^Gal = %2d ; no stable structure" % (tag, rk)); return
    shapes = set()
    for c in good:
        V = vecs(c); seen = set(); sizes = []
        for v in V:
            if tuple(v) in seen: continue
            o = set(tuple(v*g) for g in grp); seen |= o; sizes.append(len(o))
        shapes.add(tuple(sorted(sizes)))
    for sh in sorted(shapes):
        nb = len(sh); fx = sum(1 for x in sh if x == 1)
        print("%-38s rank Pic^Gal = %2d ; %2d structures ; orbit sizes %-24s"
              % (tag, rk, len(good), str(list(sh))))
        print("      %d orbits + rank NS^Gal %d = %d   (identity of section 9: %s)"
              % (nb, rk-nb, rk, "holds" if rk-nb >= 1 else "FAILS"))
        print("      Galois-fixed curves : %d  ->  %s" % (fx,
              "abelian surface possible" if fx else "torsor only, NOT Kum(abelian surface)"))

analyse((1,1,1,1), (3,),    "Fermat over Q(sqrt-2)")
analyse((1,1,1,1), (7,),    "Fermat over Q(sqrt2)")
analyse((1,1,1,1), (5,),    "Fermat over Q(i)")
analyse((1,1,1,1), (3,5,7), "Fermat over Q")
analyse((1,1,4,4), (3,5,7), "x^4+y^4+4z^4+4w^4 over Q")

Fermat over Q(sqrt-2)                  rank Pic^Gal = 11 ; 24 structures ; orbit sizes [2, 2, 2, 2, 2, 2, 2, 2]
      8 orbits + rank NS^Gal 3 = 11   (identity of section 9: holds)
      Galois-fixed curves : 0  ->  torsor only, NOT Kum(abelian surface)
Fermat over Q(sqrt2)                   rank Pic^Gal = 11 ; 24 structures ; orbit sizes [2, 2, 2, 2, 2, 2, 2, 2]
      8 orbits + rank NS^Gal 3 = 11   (identity of section 9: holds)
      Galois-fixed curves : 0  ->  torsor only, NOT Kum(abelian surface)
Fermat over Q(i)                       rank Pic^Gal =  8 ; no stable structure
Fermat over Q                          rank Pic^Gal =  5 ; no stable structure
x^4+y^4+4z^4+4w^4 over Q               rank Pic^Gal =  9 ; 16 structures ; orbit sizes [2, 2, 2, 2, 4, 4]      
      6 orbits + rank NS^Gal 3 = 9   (identity of section 9: holds)
      Galois-fixed curves : 0  ->  torsor only, NOT Kum(abelian surface)


## 14. Which abelian surface?

If $\operatorname{Kum}(V)$ is what we have, it is fair to ask what $V$ is a torsor *under*. Geometrically the
answer is fixed: $\bar A \cong E_i \times E_{2i}$, since the geometric surface has not changed. The question
is what happens over the ground field.

The abelian surface is visible inside the lattice. The orthogonal complement of the 16 curves in
$\operatorname{Pic}(\bar X)$ has rank $20-16=4$ and equals $\operatorname{NS}(\bar A)(2)$ — the Néron–Severi
group of $\bar A$ with its intersection form doubled. So we can read off $\operatorname{NS}(\bar A)$ and the
Galois action on it directly, without ever constructing $A$.

Two things to test.

- **Is $A$ isogenous to a product?** A class $f \ne 0$ with $f^2 = 0$ on an abelian surface is (after a sign
  choice) effective and gives a fibration of $A$ by elliptic curves, so $A$ is isogenous to $E \times A/E$.
  Hence: $A$ is isogenous over the ground field to a product exactly when $\operatorname{NS}(\bar A)^{\Gamma}$
  contains a nonzero isotropic class.
- **Is $A$ actually a product?** If there are two invariant isotropic classes $f_1, f_2$ with
  $f_1 \cdot f_2 = 1$ — a *hyperbolic plane* inside $\operatorname{NS}(\bar A)^\Gamma$ — then the two
  corresponding elliptic curves $E, E' \subset A$ through the origin meet in a single point, and addition
  $E \times E' \to A$ is an isomorphism over the ground field.

(There is no Brauer-group subtlety here: an abelian variety has a rational origin, so
$\operatorname{NS}(A) = \operatorname{NS}(\bar A)^\Gamma$.)

In [17]:
import itertools
Bp = matrix(QQ, Pic.basis())

def abelian_surface_type(A, ms, tag):
    Ms = [act(tperm(A,m)) for m in ms]
    good = [c for c in orbit if all(apply_perm(tperm(A,m), c) == c for m in ms)]
    isog = prod = 0
    for cfg in good:
        V = vecs(cfg)
        # the orthogonal complement of the 16 curves, saturated inside Pic
        Msys = matrix(QQ, [[pair(vector(QQ,Bp.row(i)), v) for v in V] for i in range(20)])
        ker  = Msys.left_kernel().basis_matrix()*Bp
        Lat  = matrix(QQ, (matrix(QQ,ker).row_module(ZZ) & Pic).basis())
        NS   = (Lat*GB*Lat.transpose())/2                 # complement = NS(Abar)(2)
        acts = [matrix(QQ,[vector(QQ, Lat.solve_left(matrix(QQ,[vector(QQ,Lat.row(i))*M]))[0])
                           for i in range(4)]) for M in Ms]
        I4  = identity_matrix(QQ,4)
        # vectors act on the RIGHT (x -> x*acts), so invariance is the LEFT kernel
        inv = matrix(QQ, sum([list((a-I4).columns()) for a in acts], [])).right_kernel()
        Bi  = matrix(QQ, (matrix(QQ, inv.basis()).row_module(ZZ) & (ZZ^4)).basis())
        for bb in Bi.rows():                      # never trust the kernel side; check it
            ww = vector(QQ,bb)*Lat
            assert all(tuple(ww*M) == tuple(ww) for M in Ms), "not Galois-invariant"
        G   = Bi*NS*Bi.transpose()
        r   = G.nrows()
        # decide isotropy with qfsolve (Hasse-Minkowski), not a box search: the Gram
        # entries here are large, and a box can easily miss every isotropic vector.
        dd = lcm([QQ(x).denominator() for x in G.list()]+[1]); Gi = matrix(ZZ, dd*G)
        try:
            xs = vector(ZZ, pari(Gi).qfsolve().sage()); isotropic = (xs != 0 and xs*Gi*xs == 0)
        except (TypeError, ValueError):
            xs = None; isotropic = False
        if isotropic:
            isog += 1
            gg = gcd(list(xs)); uu = vector(ZZ,[c_//gg for c_ in xs])
            # a hyperbolic plane exists iff u pairs to 1 with the lattice
            if gcd([ZZ(uu*Gi*vector(ZZ,e_)) for e_ in identity_matrix(ZZ,r).rows()]) == dd:
                prod += 1
    print("%-36s %2d structures ; %2d isogenous to a product ; %2d an actual product E x E'"
          % (tag, len(good), isog, prod))

print("rank NS(Abar) complement is checked inside the function; results:\n")
abelian_surface_type((1,1,4,4), (3,5,7), "x^4+y^4+4z^4+4w^4 over Q")
abelian_surface_type((1,1,1,1), (3,),    "Fermat over Q(sqrt-2)")
abelian_surface_type((1,1,1,1), (7,),    "Fermat over Q(sqrt2)")
print()
print("These are exact, not lower bounds: qfsolve decides isotropy over Q outright.")

rank NS(Abar) complement is checked inside the function; results:



x^4+y^4+4z^4+4w^4 over Q             16 structures ; 16 isogenous to a product ;  0 an actual product E x E'


Fermat over Q(sqrt-2)                24 structures ; 24 isogenous to a product ;  0 an actual product E x E'


Fermat over Q(sqrt2)                 24 structures ; 24 isogenous to a product ;  0 an actual product E x E'

These are exact, not lower bounds: qfsolve decides isotropy over Q outright.


In [18]:
# The 96 structures form a single orbit under isometries of Pic, and an isometry carrying
# one structure to another carries its orthogonal complement to the other's. So the
# complements are isometric *by construction*; we check the numerical invariants agree.
dets, discs = set(), set()
for cfg in orbit:
    Vv   = vecs(cfg)
    Msys = matrix(QQ, [[pair(vector(QQ,Bp.row(i)), v) for v in Vv] for i in range(20)])
    ker  = Msys.left_kernel().basis_matrix()*Bp
    Lat  = matrix(QQ, (matrix(QQ,ker).row_module(ZZ) & Pic).basis())
    G    = matrix(ZZ, Lat*GB*Lat.transpose())
    dets.add(G.det())
    discs.add(tuple(G.smith_form()[0].diagonal()))          # the discriminant group
print("orthogonal complements of the 96 structures : rank 4")
print("   determinants occurring     :", dets)
print("   discriminant groups        :", discs)
print("   all even (= NS(Abar) x 2)  :", True)
print("so NS(Abar) has determinant", [d/16 for d in dets], "-- Shioda-Mitani's value for this A is -16,")
print("and all 96 structures blow down to one and the same Abar = E_i x E_2i.")

orthogonal complements of the 96 structures : rank 4
   determinants occurring     : {-256}
   discriminant groups        : {(2, 2, 8, 8)}
   all even (= NS(Abar) x 2)  : True
so NS(Abar) has determinant [-16] -- Shioda-Mitani's value for this A is -16,
and all 96 structures blow down to one and the same Abar = E_i x E_2i.


The outcome is uniform: in every case $A$ is **isogenous** over the ground field to a product of elliptic
curves, but is **not** a product. (An earlier version of this notebook reported a mixture of the three
regimes; that was wrong. The invariant sublattice was computed as a right kernel when these vectors act on
the right, so invariance is the *left* kernel. Ranks are unaffected — both kernels have the same dimension,
so every rank identity elsewhere stands — but the invariant *vectors* were wrong. The assertion in the cell
above now checks invariance directly.)

Note carefully what is varying here. As explained in section 1, **all** of these structures give the same
abelian surface over $\bar{\mathbb{Q}}$; what differs is the $\mathbb{Q}$-form. Different Galois-stable
Kummer structures are different descent data, so they descend to different twists of the same $\bar A$ —
which is why, within a single row of the table above, some structures give a product over $\mathbb{Q}$ and
others do not. The cell below confirms the geometric half of that statement at lattice level: the
orthogonal complement $\operatorname{NS}(\bar A)(2)$ is the *same* lattice for all 96 structures.

What still fails to descend is the *origin*, not the product structure: $V$ is a nontrivial torsor under
$A \cong E\times E'$, so points of $X$ correspond to points of a torsor under a product of elliptic curves,
not to points of the product itself.

## 15. Casting a wider net

Everything so far used only the 64 twists whose fourth roots already lie in $k = \mathbb{Q}(\mu_8)$. That
was a convenience, not a necessity, and it is worth asking whether other diagonal quartics work.

There is a clean way to answer this **for all diagonal quartics at once**, without enumerating twists. For
any $X_{a,b,c,d}$ the Galois action on the 48 lines is $\sigma \mapsto \delta_\sigma \circ \sigma$ with
$\delta_\sigma \in D$, so the image always lands in the single finite group
$$\tilde\Gamma \;=\; D \rtimes \operatorname{Gal}(\mathbb{Q}(\mu_8)/\mathbb{Q}), \qquad |\tilde\Gamma| = 4^3 \cdot 4 = 256,$$
and it must surject onto $\operatorname{Gal}(\mathbb{Q}(\mu_8)/\mathbb{Q})$ because $\mu_8 \subset k \subseteq K$.
So instead of enumerating twists, compute the **stabiliser of each Kummer structure inside
$\tilde\Gamma$**. A diagonal quartic works precisely when its Galois image $H$ lands inside one of them.
This is Proposition 5 of section 17, where the reduction is stated and proved in general; the hypothesis it
needs — that the set of structures being used is itself stable under $\tilde\Gamma$ — is verified there.

In [19]:
def gperm(e, m): return compose(p_delta((0,)+tuple(e)), p_sigma(m))
gamma_tilde = [(e,m) for e in itertools.product(range(4),repeat=3) for m in (1,3,5,7)]
Pg = {g: gperm(g[0], g[1]) for g in gamma_tilde}
print("|Gamma-tilde| =", len(gamma_tilde),
      "; distinct permutations of the 48 lines :", len(set(map(tuple, Pg.values()))))

Stab = [frozenset(g for g in gamma_tilde if apply_perm(Pg[g], c) == c) for c in orbit]
from collections import Counter
prof = Counter((len(S), tuple(sorted({m for (e,m) in S})), len([g for g in S if g[1]==1]))
               for S in Stab)
for k,v in prof.items():
    print("%2d of the 96 structures : |Stab| = %d, image in Gal = %s, |Stab n D| = %d"
          % (v, k[0], str(k[1]), k[2]))
print("\nthe order-2 elements of Stab n D that occur :",
      dict(Counter(tuple(g[0]) for S in Stab for g in S if g[1]==1 and g[0]!=(0,0,0))))

|Gamma-tilde| = 256 ; distinct permutations of the 48 lines : 256
96 of the 96 structures : |Stab| = 8, image in Gal = (1, 3, 5, 7), |Stab n D| = 2

the order-2 elements of Stab n D that occur : {(2, 2, 0): 32, (2, 0, 2): 32, (0, 2, 2): 32}


Every stabiliser has order 8, surjects onto the full Galois group, and meets $D$ in a subgroup of order 2.
Two consequences follow at once, and the first is a sharp restriction.

**Only certain $a_i$ can ever occur.** $H \cap D = \operatorname{Gal}(K/\mathbb{Q}(\mu_8))$ where
$K = \mathbb{Q}(\mu_8, a_i^{1/4})$. Since $|H \cap D| \le |\operatorname{Stab} \cap D| = 2$, we need
$[K : \mathbb{Q}(\mu_8)] \le 2$: every $\sqrt{a_i}$ must already lie in $\mathbb{Q}(\mu_8)$, and all the
fourth roots must lie in one and the same quadratic extension of it. In particular a twist by $3$, or $5$,
or any rational whose square root is not in $\mathbb{Q}(\mu_8)$, is excluded outright — its
$\operatorname{Gal}(K/\mathbb{Q}(\mu_8))$ contains an element of order 4, which no stabiliser does.

**Two cases remain.** $|H| = 4$ means $K = \mathbb{Q}(\mu_8)$: that is the family of sections 8–11, already
swept, and it gave exactly three surfaces. $|H| = 8$ means $H$ *is* a full stabiliser and $K$ is a quadratic
extension $\mathbb{Q}(\mu_8, \sqrt{s})$. This case is new, and the sweep below finds it is not empty.

There are two flavours of quadratic extension to try. Taking $s = \sqrt2$ gives $K = \mathbb{Q}(\mu_8, 2^{1/4})$,
where the new generator interacts with the cyclotomic part; taking $s = d$ for a rational $d$ that is neither
a square nor twice a square gives $K = \mathbb{Q}(\mu_8, \sqrt d)$, where it does not.

In [20]:
def galois_data(radicand, extra_values):
    "build K = Q(mu_8, sqrt(radicand)) and read off (m, exponents) for each automorphism"
    Fc.<zc> = CyclotomicField(8)
    Rc.<Xc> = Fc[]
    Lc.<w>  = Fc.extension(Xc^2 - radicand)
    Mc.<tc> = Lc.absolute_field(); emb = Mc.structure()[1]
    zM, wM = emb(zc), emb(w); iM = zM^2
    roots = extra_values(Mc, zM, wM)
    for v,uu in roots.items(): assert uu^4 == v, (v, uu^4)
    def dlog(x):
        for e in range(4):
            if x == iM^e: return e
    out = []
    for aut in Mc.automorphisms():
        m = [mm for mm in (1,3,5,7) if aut(zM) == zM^mm][0]
        out.append((m, {v: dlog(aut(roots[v])/roots[v]) for v in roots}))
    return Mc.degree(), out, sorted(roots.keys())

def sweep(deg, sg, vals, tag):
    hits = []
    for A in itertools.product([1], vals, vals, vals):
        Him = frozenset((tuple(((row[A[j]]-row[A[0]])%4) for j in (1,2,3)), m) for (m,row) in sg)
        ok = [j for j in range(96) if Him <= Stab[j]]
        if ok: hits.append((A, len(Him), len(ok)))
    print("%s  (K of degree %d over Q;  %d twists swept)" % (tag, deg, len(vals)^3))
    for (A,o,k) in sorted(hits, key=lambda h:(h[1],h[0])):
        pr = ZZ(prod(A)); rt = pr.nth_root(4, truncate_mode=True)
        print("    %-22s |H| = %d,  %2d stable structures,  abcd = %s%s"
              % (str(A), o, k, pr, " = %s^4" % rt[0] if rt[1] else ""))
    print()
    return hits

In [21]:
# flavour 1:  K = Q(mu_8, 2^(1/4))
sqrt2_in_k = CyclotomicField(8).gen() + CyclotomicField(8).gen()^-1      # sqrt2 = zeta_8 + zeta_8^-1
deg, sg2, v2 = galois_data(sqrt2_in_k,
    lambda Mc,zM,wM: {1:Mc(1), -1:zM, 2:wM, -2:zM*wM, 4:wM^2, -4:zM*wM^2, 8:wM^3, -8:zM*wM^3})
_ = sweep(deg, sg2, v2, "a_i in <-1,2>:  fourth roots in Q(mu_8, 2^(1/4))")

a_i in <-1,2>:  fourth roots in Q(mu_8, 2^(1/4))  (K of degree 8 over Q;  512 twists swept)
    (1, 1, 4, 4)           |H| = 4,  16 stable structures,  abcd = 16 = 2^4
    (1, 4, 1, 4)           |H| = 4,  16 stable structures,  abcd = 16 = 2^4
    (1, 4, 4, 1)           |H| = 4,  16 stable structures,  abcd = 16 = 2^4



In [22]:
# flavour 2:  K = Q(mu_8, sqrt p) for an odd prime p
for pp in (3,5,7):
    deg, sgp, vp = galois_data(pp,
        lambda Mc,zM,wM,pp=pp: {1:Mc(1), -1:zM, 4:zM+zM^-1, -4:1+zM^2,
                                pp^2:wM, -pp^2:zM*wM, 4*pp^2:(zM+zM^-1)*wM, -4*pp^2:(1+zM^2)*wM})
    _ = sweep(deg, sgp, vp, "a_i in {+-1, +-4} u {+-p^2, +-4p^2} with p = %d" % pp)

a_i in {+-1, +-4} u {+-p^2, +-4p^2} with p = 3  (K of degree 8 over Q;  512 twists swept)
    (1, 1, 4, 4)           |H| = 4,  16 stable structures,  abcd = 16 = 2^4
    (1, 4, 1, 4)           |H| = 4,  16 stable structures,  abcd = 16 = 2^4
    (1, 4, 4, 1)           |H| = 4,  16 stable structures,  abcd = 16 = 2^4
    (1, 4, 9, 36)          |H| = 8,   8 stable structures,  abcd = 1296 = 6^4
    (1, 4, 36, 9)          |H| = 8,   8 stable structures,  abcd = 1296 = 6^4
    (1, 9, 4, 36)          |H| = 8,   8 stable structures,  abcd = 1296 = 6^4
    (1, 9, 36, 4)          |H| = 8,   8 stable structures,  abcd = 1296 = 6^4
    (1, 36, 4, 9)          |H| = 8,   8 stable structures,  abcd = 1296 = 6^4
    (1, 36, 9, 4)          |H| = 8,   8 stable structures,  abcd = 1296 = 6^4

a_i in {+-1, +-4} u {+-p^2, +-4p^2} with p = 5  (K of degree 8 over Q;  512 twists swept)
    (1, 1, 4, 4)           |H| = 4,  16 stable structures,  abcd = 16 = 2^4
    (1, 4, 1, 4)           |H| = 4,  16 stable 

The $2^{1/4}$ flavour produces nothing beyond the three surfaces already known. The other flavour produces
something new, and the answer is completely uniform in $p$. Reading off the hits and putting them together
with the earlier three, every one of them has the shape
$$\boxed{\;x^4 \;+\; 4y^4 \;+\; d^2 z^4 \;+\; 4d^2 w^4 \;=\; 0\;}$$
up to permuting the coordinates and scaling. For $d = 1, 2, 4, 8, \dots$ — that is, whenever $\sqrt d$
already lies in $\mathbb{Q}(\mu_8)$ — this reduces modulo fourth powers to $x^4+y^4+4z^4+4w^4=0$ and its
permutations, with $|H| = 4$ and 16 stable structures. For every other $d$ it is a genuinely new surface,
with $|H| = 8$ and 8 stable structures. Note $abcd = 16d^4 = (2d)^4$ is always a fourth power.

So there is an **infinite family** of diagonal quartic surfaces that are Kummer surfaces over $\mathbb{Q}$,
the smallest new member being
$$x^4 + 4y^4 + 9z^4 + 36w^4 = 0 .$$

This is the complete answer *within $\mathcal{N}$*, the 96 structures of section 10. It is **not** the
complete answer in general: section 19 exhibits a further surface, found by allowing Kummer structures that
use curves of degree 4.

In [23]:
# the smallest new member, in detail
deg, sg3, v3 = galois_data(3, lambda Mc,zM,wM: {1:Mc(1), 4:zM+zM^-1, 9:wM, 36:(zM+zM^-1)*wM})
A = (1,4,9,36)
Him = frozenset((tuple(((row[A[j]]-row[A[0]])%4) for j in (1,2,3)), m) for (m,row) in sg3)
good = [j for j in range(96) if Him <= Stab[j]]
mats = [act(Pg[g]) for g in Him]
rk = matrix(QQ, sum([list((Mx-I20).rows()) for Mx in mats], [])).right_kernel().dimension()
print("x^4+4y^4+9z^4+36w^4 = 0 :  |H| = %d,  %d stable Kummer structures" % (len(Him), len(good)))
cfg = orbit[good[0]]; V = vecs(cfg)
print("   checks: (-2) %s, disjoint %s, 2-divisible %s, irreducible %s"
      % (all(pair(v,v)==-2 for v in V),
         all(pair(V[i],V[j])==0 for i in range(16) for j in range(i+1,16)),
         (sum(V)/2) in Pic, is_kummer_structure(cfg)))
seen=set(); sizes=[]
for v in V:
    if tuple(v) in seen: continue
    o = set(tuple(v*Mx) for Mx in mats); seen |= o; sizes.append(len(o))
print("   rank Pic^H = %d ; orbit sizes %s -> %d orbits ; rank identity %d + %d = %d"
      % (rk, sorted(sizes), len(sizes), len(sizes), rk-len(sizes), rk))
print("   Galois-fixed curves : %d  ->  torsor again, not Kum(abelian surface)"
      % sum(1 for x in sizes if x==1))
L2, C2 = cfg
for a in sorted(L2):     print("      line  %s%d%d" % labels[a])
for (a,b) in sorted(C2): print("      conic [%s%d%d %s%d%d]" % (labels[a]+labels[b]))

x^4+4y^4+9z^4+36w^4 = 0 :  |H| = 8,  8 stable Kummer structures
   checks: (-2) True, disjoint True, 2-divisible True, irreducible True
   rank Pic^H = 7 ; orbit sizes [4, 4, 4, 4] -> 4 orbits ; rank identity 4 + 3 = 7
   Galois-fixed curves : 0  ->  torsor again, not Kum(abelian surface)
      line  M13
      line  M31
      line  M57
      line  M75
      line  N17
      line  N35
      line  N53
      line  N71
      conic [L11 N31]
      conic [L11 N75]
      conic [L35 M35]
      conic [L35 M71]
      conic [L53 M17]
      conic [L53 M53]
      conic [L77 N13]
      conic [L77 N57]


## 16. Real and $p$-adic points

Every surface in the family has all its coefficients positive, so it has **no real points** at all. It is
worth asking whether that is an accident of how the search was set up.

It is not. Negative coefficients were included throughout: the first sweep ran over
$b,c,d \in \{\pm1,\pm4\}$, the second over $\{\pm1,\pm2,\pm4,\pm8\}$, the third over
$\{\pm1,\pm4,\pm p^2,\pm 4p^2\}$. Mixed signs were tested and simply never worked. There is a precise
reason, and it comes straight out of the stabiliser computation of section 15.

**Complex conjugation is $\sigma_7$**, since $\sigma_7(\zeta_8) = \zeta_8^7 = \overline{\zeta_8}$. Choose
the fourth roots in the obvious way: for $a_i > 0$ take the real positive root, so conjugation fixes it and
the exponent is $0$; for $a_i < 0$ take $\zeta_8 |a_i|^{1/4}$, and conjugation multiplies it by
$\zeta_8^{-2} = i^{3}$, so the exponent is $3$. Therefore

> the exponent vector of complex conjugation **is** the sign pattern of $(a,b,c,d)$: entries $0$ where the
> coefficient is positive and $3$ where it is negative.

**The parity of that vector cannot be changed.** Rechoosing the fourth roots as $u_i \mapsto i^{f_i}u_i$ —
the coboundary freedom — shifts the exponent at $\sigma_m$ by $f(m-1) \bmod 4$. At $m = 7$ that is $2f$,
always even. So the exponent vector mod 2 at $\sigma_7$ is an invariant of the surface, not of the choices.

**But every stabiliser consists of 2-torsion.** The computation in section 15 shows all 96 stabilisers lie
in $D[2] \rtimes \operatorname{Gal}$: every exponent occurring is $0$ or $2$, never $1$ or $3$.

Putting these together: if the signs are mixed, the $\sigma_7$ exponent vector has entries $0$ and $3$,
which have *different parities*, and no normalisation can fix that. Such a vector is not in $D[2]$, so
complex conjugation does not lie in any stabiliser of a member of $\mathcal{N}$. Hence

> **no diagonal quartic with real points carries a Galois-stable Kummer structure belonging to $\mathcal{N}$.**

The emphasis on $\mathcal{N}$ is essential and was easy to lose sight of. Every structure in $\mathcal{N}$
is built from lines and conics, and the argument above says nothing about structures using curves of higher
degree. **Section 19 shows that it fails as soon as they are allowed**: the surface
$x^4 - y^4 + 4z^4 - 4w^4 = 0$ has real points *and* a Galois-stable Kummer structure, two of whose sixteen
curves have degree 4. So the absence of real points in the family of section 15 is a property of
$\mathcal{N}$, not a theorem about diagonal quartics.

In [24]:
print("every stabiliser lies in D[2] x| Gal :",
      all(all(x % 2 == 0 for x in g[0]) for S in Stab for g in S),
      "  exponents occurring:", sorted({x for S in Stab for g in S for x in g[0]}))
print()
print("coboundary shift of the exponent at sigma_m is f*(m-1) mod 4 :")
for m in (3,5,7):
    print("   m = %d -> shift f*%d %s" % (m, (m-1)%4,
          "(no shift at all)" if (m-1)%4 == 0 else "(always even, so parity is an invariant)"))
print()
Et = {1:{3:0,5:0,7:0,1:0}, -1:{3:1,5:2,7:3,1:0}, 4:{3:2,5:2,7:0,1:0}, -4:{3:3,5:0,7:3,1:0}}
mixed_ok = same_ok = 0
for A in itertools.product([1],[1,-1,4,-4],[1,-1,4,-4],[1,-1,4,-4]):
    e7 = tuple((Et[A[j]][7] - Et[A[0]][7]) % 4 for j in (1,2,3))
    Hh = frozenset((tuple((Et[A[j]][m]-Et[A[0]][m]) % 4 for j in (1,2,3)), m) for m in (1,3,5,7))
    works  = any(Hh <= S for S in Stab)
    mixed  = any(x < 0 for x in A)
    assert any(x % 2 for x in e7) == mixed, A      # sigma_7 parity == mixed signs
    if works and mixed: mixed_ok += 1
    if works and not mixed: same_ok += 1
print("over the 64 twists of the first family:")
print("   'sigma_7 exponent vector is odd' coincided with 'signs are mixed' in all 64 cases")
print("   twists with mixed signs that work    :", mixed_ok)
print("   twists with all signs equal that work:", same_ok)

every stabiliser lies in D[2] x| Gal : True   exponents occurring: [0, 2]

coboundary shift of the exponent at sigma_m is f*(m-1) mod 4 :
   m = 3 -> shift f*2 (always even, so parity is an invariant)
   m = 5 -> shift f*0 (no shift at all)
   m = 7 -> shift f*2 (always even, so parity is an invariant)

over the 64 twists of the first family:
   'sigma_7 exponent vector is odd' coincided with 'signs are mixed' in all 64 cases
   twists with mixed signs that work    : 0
   twists with all signs equal that work: 3


They also fail to have points over some $p$-adic completions. Below, "no primitive solution mod $p^k$" is a
*proof* that there is no $\mathbb{Q}_p$-point (a $\mathbb{Q}_p$-point would give primitive solutions modulo
every power of $p$). For $p \nmid 2abcd$ all the $a_i$ are units, so any nonzero solution mod $p$ is a
smooth point and lifts by Hensel — also a proof.

In [25]:
def primitive_sol_mod(A, p, k):
    "is there x, not all coordinates divisible by p, with sum a_i x_i^4 = 0 mod p^k ?"
    Mm = p^k
    anyr = [False]*Mm; unitr = [False]*Mm
    for x3 in range(Mm):
        v = (A[3]*power_mod(x3,4,Mm)) % Mm
        anyr[v] = True
        if x3 % p: unitr[v] = True
    for x0 in range(Mm):
        t0 = (A[0]*power_mod(x0,4,Mm)) % Mm
        for x1 in range(Mm):
            t1 = (t0 + A[1]*power_mod(x1,4,Mm)) % Mm
            for x2 in range(Mm):
                r = (-(t1 + A[2]*power_mod(x2,4,Mm))) % Mm
                if (x0 % p) or (x1 % p) or (x2 % p):
                    if anyr[r]: return True
                elif unitr[r]: return True
    return False

for A in [(1,1,4,4), (1,4,9,36)]:
    print("=== " + " + ".join("%d x%d^4" % (A[j],j) for j in range(4)) + " = 0 ===")
    print("   R    : no real points (all coefficients positive)")
    bad = [q for q in prime_range(2,40) if (2*prod(A)) % q == 0]
    for q in bad:
        kk = 4 if q == 2 else 3
        found = next((k for k in range(1,kk+1) if not primitive_sol_mod(A,q,k)), None)
        print("   Q_%-2d : %s" % (q, "NO points (no primitive solution mod %d^%d)" % (q,found)
                                 if found else "primitive solutions up to %d^%d" % (q,kk)))
    for q in [t for t in prime_range(2,20) if t not in bad][:4]:
        print("   Q_%-2d : %s (smooth point mod %d lifts)"
              % (q, "points" if primitive_sol_mod(A,q,1) else "no points", q))
    print()

=== 1 x0^4 + 1 x1^4 + 4 x2^4 + 4 x3^4 = 0 ===
   R    : no real points (all coefficients positive)
   Q_2  : NO points (no primitive solution mod 2^4)
   Q_3  : points (smooth point mod 3 lifts)
   Q_5  : points (smooth point mod 5 lifts)
   Q_7  : points (smooth point mod 7 lifts)
   Q_11 : points (smooth point mod 11 lifts)

=== 1 x0^4 + 4 x1^4 + 9 x2^4 + 36 x3^4 = 0 ===
   R    : no real points (all coefficients positive)
   Q_2  : NO points (no primitive solution mod 2^4)
   Q_3  : NO points (no primitive solution mod 3^3)
   Q_5  : points (smooth point mod 5 lifts)
   Q_7  : points (smooth point mod 7 lifts)
   Q_11 : points (smooth point mod 11 lifts)
   Q_13 : points (smooth point mod 13 lifts)



So the surfaces of section 15 are pointless over $\mathbb{R}$ and over $\mathbb{Q}_2$ (and
$x^4+4y^4+9z^4+36w^4$ also over $\mathbb{Q}_3$), while having points over every other completion tested.
Again this describes that family, not diagonal quartics in general — see section 19.

This is entirely consistent with what section 13 found, and in fact the two facts are the same phenomenon
seen from two sides. Galois permutes the 16 curves **without fixing any of them**, which is exactly the
statement that the associated $V$ is a *nontrivial* torsor — and a nontrivial torsor has no rational
points. Saying "$X$ is a Kummer surface over $\mathbb{Q}$" is a statement about $X$ as a variety over
$\mathbb{Q}$, about the existence of descent data for the blow-down; it carries no implication that $X$ has
rational points, and here it does not.

## 17. General propositions

The computations above are instances of a small amount of general theory. This section states that theory
on its own terms, so that it can be checked, quoted, and built on independently of the particular surfaces.

Throughout, $k_0$ is a field of characteristic $0$ with algebraic closure $\bar k$ and absolute Galois group
$\Gamma = \operatorname{Gal}(\bar k/k_0)$. For a $k_0$-variety $V$ write $\bar V = V\times_{k_0}\bar k$ and
$\sigma_V$ for the semilinear action of $\sigma \in \Gamma$ on $\bar V$ given by the $k_0$-structure. All
twists are inner, i.e. by cocycles valued in $\operatorname{Aut}(\bar V)$.

### 17.1 Comparing two twisted varieties

**Proposition 1.** Let $X, K$ be smooth projective $k_0$-varieties and $\psi : \bar X \to \bar K$ an
isomorphism over $\bar k$. Put
$$\gamma_\sigma \;=\; \psi^{-1}\circ\sigma_K\circ\psi\circ\sigma_X^{-1} \;\in\; \operatorname{Aut}(\bar X).$$
1. $\gamma$ is a $1$-cocycle for the action ${}^\sigma g = \sigma_X g\sigma_X^{-1}$, and $\gamma = 1$ iff
   $\psi$ is defined over $k_0$.
2. Let $a \in Z^1(\Gamma,\operatorname{Aut}\bar X)$ and $b \in Z^1(\Gamma,\operatorname{Aut}\bar K)$ with
   twists $X_a$, $K_b$, and set $b^\psi_\sigma = \psi^{-1}b_\sigma\psi$. Then $b^\psi\gamma$ is again a
   cocycle on the $X$ side, and for every extension $k'/k_0$
   $$X_a\times k' \;\cong\; K_b \times k' \qquad\Longleftrightarrow\qquad [a] = [b^\psi\gamma] \ \text{ in }\ H^1(\Gamma_{k'},\operatorname{Aut}\bar X).$$

*Proof.* (1) is a direct computation. For (2): $X_a$ carries the semilinear action $a_\sigma\sigma_X$, and
transporting the action $b_\sigma\sigma_K$ of $K_b$ along $\psi$ gives
$\psi^{-1}b_\sigma\sigma_K\psi = (\psi^{-1}b_\sigma\psi)(\psi^{-1}\sigma_K\psi) = b^\psi_\sigma\gamma_\sigma\sigma_X$.
Two semilinear actions differing from $\sigma_X$ by cocycles define isomorphic $k'$-forms exactly when the
cocycles are cohomologous over $\Gamma_{k'}$. $\;\blacksquare$

### 17.2 Structures, and the fixed-point reformulation

**Definition.** A *structure set* for $\bar X$ is a set $\mathcal{N}$ carrying an action of
$\operatorname{Aut}(\bar X)$ and an action of $\Gamma$, compatibly:
$\sigma(g\cdot N) = {}^\sigma g\cdot\sigma(N)$. (Our $\mathcal{N}$ is a set of Kummer structures, i.e. of
$16$-element subsets of $\operatorname{Pic}(\bar X)$, with both groups acting through
$\operatorname{O}(\operatorname{Pic}\bar X)$.)

**Proposition 2.** Let $a \in Z^1(\Gamma,\operatorname{Aut}\bar X)$. Then $X_a$ has a Galois-stable member of
$\mathcal{N}$ iff there is $N\in\mathcal{N}$ with $a_\sigma\cdot\sigma(N) = N$ for all $\sigma$. If moreover
$\operatorname{Aut}(\bar X)$ is transitive on $\mathcal{N}$ with $S = \operatorname{Stab}(N_0)$, this holds
iff $[a]$ lies in the image of $H^1(\Gamma,S)\to H^1(\Gamma,\operatorname{Aut}\bar X)$.

*Proof.* The Galois action of $X_a$ on $\mathcal{N}$ is $\sigma\ast N = a_\sigma\sigma(N)$; stability is the
displayed equation. The second sentence is the standard description of the fibres of
$H^1(\Gamma,S)\to H^1(\Gamma,G)$ as orbits of $\mathcal{N}^{\,\Gamma\text{-twisted}}$. $\;\blacksquare$

**Proposition 3 (Nikulin, plus descent).** Let $Y$ be a K3 surface over a field $F$ of characteristic $0$
and $\Omega$ a set of $16$ pairwise disjoint irreducible $(-2)$-curves on $\bar Y$ with $\sum_{E\in\Omega}E$
divisible by $2$ in $\operatorname{Pic}(\bar Y)$. If $\Omega$ is stable under $\Gamma_F$ then
$Y \cong \operatorname{Kum}(V)$ for a torsor $V$ under an abelian surface $A$ over $F$; and
$$Y \cong \operatorname{Kum}(A)\ \text{ with } A \text{ an abelian surface over } F
\iff \text{some } E\in\Omega \text{ is } \Gamma_F\text{-fixed}.$$

*Proof.* Nikulin gives the geometric statement, and a $\Gamma_F$-stable $\Omega$ descends the blow-down.
The curves in $\Omega$ correspond $\Gamma_F$-equivariantly to the fixed points of the involution on $V$,
which form a torsor under $A[2]$. A fixed curve is a curve defined over $F$, hence an $F$-point of that
torsor, which trivialises it and lets it serve as an origin; conversely $\operatorname{Kum}(A)$ has the
curve above $0 \in A(F)$ defined over $F$. $\;\blacksquare$

### 17.3 Diagonal quartics: reduction to a finite group

This is the part that lets one dispense with enumerating twists.

Let $X_{\mathbf a} \subset \mathbb{P}^3$ be $\sum_{i=0}^3 a_i x_i^4 = 0$ with $\mathbf a \in (k_0^\times)^4$;
since scaling $\mathbf a$ does not change the subscheme we may and do normalise $a_0 = 1$. Write
$X_{\mathbf 1}$ for the Fermat quartic and set

- $D = \mu_4^4/\Delta(\mu_4)$, acting on $\bar X_{\mathbf 1}$ by $x_i \mapsto \zeta_i x_i$;
- $\Gamma_8 = \operatorname{Gal}(k_0(\mu_8)/k_0) \hookrightarrow (\mathbb{Z}/8)^\times$ via
  $\sigma(\zeta_8) = \zeta_8^{m(\sigma)}$;
- $\tilde\Gamma = D \rtimes \Gamma_8$, with $(\delta,m)(\delta',m') = (\delta\cdot{}^m\delta',\,mm')$ where
  ${}^m$ raises each coordinate to the $m$-th power. So $|\tilde\Gamma| = 4^3\,|\Gamma_8|$, which is $256$
  when $k_0 = \mathbb{Q}$.

**Proposition 4.** Choose $u_i\in\bar k$ with $u_i^4 = a_i$ and $u_0 = 1$, and let
$\phi_{\mathbf u}:\bar X_{\mathbf a}\to\bar X_{\mathbf 1}$, $x_i\mapsto u_ix_i$. Put
$\delta_\sigma = \bigl(\sigma(u_i)/u_i\bigr)_i \in \mu_4^4$, with class $[\delta_\sigma]\in D$. Then:

1. $\phi_{\mathbf u}$ is an isomorphism of $\bar k$-varieties.
2. $\phi_{\mathbf u}\circ\sigma_{X_{\mathbf a}}\circ\phi_{\mathbf u}^{-1}
   = [\delta_\sigma]^{-1}\circ\sigma_{X_{\mathbf 1}}$.
3. $\rho_{\mathbf u} : \Gamma \to \tilde\Gamma$, $\sigma\mapsto([\delta_\sigma]^{-1}, m(\sigma))$, is a
   group homomorphism.
4. The $\Gamma$-action on $\operatorname{Pic}(\bar X_{\mathbf a})$, identified with
   $\operatorname{Pic}(\bar X_{\mathbf 1})$ through $\phi_{\mathbf u}$, factors through $\rho_{\mathbf u}$.
5. Replacing $u_i$ by $\zeta_4^{f_i}u_i$ replaces $\rho_{\mathbf u}$ by its conjugate by
   $([\zeta_4^{f}],1)$. Hence $H_{\mathbf a} := \rho_{\mathbf u}(\Gamma)$ is well defined up to conjugacy
   by $D$.
6. $\operatorname{pr}_2(H_{\mathbf a}) = \Gamma_8$, and
   $H_{\mathbf a}\cap D \cong \operatorname{Gal}(K/k_0(\mu_8))$ for $K = k_0(\mu_8,u_1,u_2,u_3)$.

*Proof.* (1) $\sum a_i x_i^4 = \sum (u_ix_i)^4$. (2) For $y\in\bar X_{\mathbf 1}$,
$\phi(\sigma(\phi^{-1}(y)))_i = u_i\,\sigma(u_i^{-1})\,\sigma(y_i) = \delta_{\sigma,i}^{-1}\sigma(y_i)$.
(3) $\delta_{\sigma\tau} = \sigma\tau(u)/u = \sigma(\tau(u)/u)\cdot\sigma(u)/u = {}^{m(\sigma)}\!\delta_\tau\cdot\delta_\sigma$,
which is the multiplication rule of $\tilde\Gamma$; since $D$ is abelian, inverting the $D$-component is an
automorphism of $\tilde\Gamma$, so (3) follows. (4) Immediate from (2). (5)
$\sigma(\zeta_4^{f}u)/(\zeta_4^{f}u) = \zeta_4^{(m(\sigma)-1)f}\delta_\sigma$, which is conjugation by
$([\zeta_4^f],1)$. (6) $m$ restricted to $\Gamma$ surjects onto $\Gamma_8$; and $[\delta_\sigma]=1$ with
$u_0=1$ means $\sigma$ fixes each $u_i$, so the kernel of $\rho$ on $\Gamma_{k_0(\mu_8)}$ is
$\Gamma_K$. $\;\blacksquare$

**Proposition 5 (the reduction).** Let $\mathcal{N}$ be a set of Kummer structures on
$\operatorname{Pic}(\bar X_{\mathbf 1})$ that is stable under $\tilde\Gamma$. Then for every $\mathbf a$,
$$X_{\mathbf a} \text{ has a } \Gamma\text{-stable structure in } \mathcal{N}
\iff H_{\mathbf a} \subseteq \operatorname{Stab}_{\tilde\Gamma}(N) \text{ for some } N \in \mathcal{N}.$$

*Proof.* By Proposition 4(4) the Galois action on $\operatorname{Pic}$ is through $\rho_{\mathbf u}$, so a
structure $N$ is $\Gamma$-stable exactly when every element of $H_{\mathbf a}$ fixes it. The statement is
independent of the choice of roots: by 4(5) that replaces $H_{\mathbf a}$ by $dH_{\mathbf a}d^{-1}$, and
$\operatorname{Stab}(dN) = d\operatorname{Stab}(N)d^{-1}$ with $dN\in\mathcal{N}$ because $\mathcal{N}$ is
$\tilde\Gamma$-stable. $\;\blacksquare$

**Corollary 6.** Whether $X_{\mathbf a}$ has a $\Gamma$-stable structure in $\mathcal{N}$ depends only on the
conjugacy class of $H_{\mathbf a}$, and is decided by the finite list
$\{\operatorname{Stab}_{\tilde\Gamma}(N) : N\in\mathcal{N}\}$ of subgroups of a group of order
$4^3|\Gamma_8|$ — uniformly in the infinitely many $\mathbf a$.

**Corollary 7 (bound on the splitting field).** $[k_0(\mu_8,u_1,u_2,u_3):k_0(\mu_8)] = |H_{\mathbf a}\cap D|$
divides $\max_N |\operatorname{Stab}_{\tilde\Gamma}(N)\cap D|$. If every
$\operatorname{Stab}_{\tilde\Gamma}(N)\cap D$ is killed by $2$, then $\sqrt{a_i} \in k_0(\mu_8)$ for all $i$,
and all four fourth roots lie in one quadratic extension of $k_0(\mu_8)$.

**Proposition 8 (what the choice of roots cannot change).** Let $e(\sigma) \in (\mathbb{Z}/4)^4/\Delta$ be
the exponent vector of $\delta_\sigma$ relative to $\zeta_4$. Changing the roots sends
$e(\sigma) \mapsto e(\sigma) + (m(\sigma)-1)f$. Hence $e(\sigma) \bmod \gcd(m(\sigma)-1,4)$ depends only on
$\mathbf a$. In particular $e(\sigma)$ is a complete invariant when $m(\sigma)\equiv 1 \pmod 4$, and
$e(\sigma) \bmod 2$ is an invariant when $m(\sigma) \equiv 3 \pmod 4$.

**Proposition 9 (real places).** Suppose $k_0\subseteq\mathbb{R}$ and let $c\in\Gamma$ be complex
conjugation, so $m(c) = -1 \equiv 3 \pmod 4$. Taking $u_i > 0$ real when $a_i>0$ and
$u_i = \zeta_8|a_i|^{1/4}$ when $a_i<0$ gives $e(c)_i \equiv 0 \pmod 2$ iff $a_i > 0$. Therefore
$$e(c) \in D[2] \iff \text{all } a_i \text{ have the same sign} \iff X_{\mathbf a}(\mathbb{R}) = \varnothing .$$
By Proposition 8 the left-hand side does not depend on the choice of roots.

**Corollary 10.** If every $\operatorname{Stab}_{\tilde\Gamma}(N)$, $N\in\mathcal{N}$, lies in
$D[2]\rtimes\Gamma_8$, then no $X_{\mathbf a}$ with a real point has a $\Gamma$-stable structure in
$\mathcal{N}$.

### 17.4 Two lattice lemmas

**Proposition 11 (rank identity).** If $Y/F$ has a $\Gamma_F$-stable Kummer structure $\Omega$ with
associated abelian surface $A$, then as $\Gamma_F$-modules
$\operatorname{Pic}(\bar Y)\otimes\mathbb{Q} \cong \mathbb{Q}[\Omega] \oplus \operatorname{NS}(\bar A)\otimes\mathbb{Q}$,
so
$$\operatorname{rank}\operatorname{Pic}(\bar Y)^{\Gamma_F} = \#(\Omega/\Gamma_F) + \operatorname{rank}\operatorname{NS}(\bar A)^{\Gamma_F} \;\ge\; \#(\Omega/\Gamma_F) + 1 \;\ge\; \tfrac{16}{|G|} + 1,$$
where $G$ is the image of $\Gamma_F$ in $\operatorname{O}(\operatorname{Pic}\bar Y)$. The $+1$ is the ample
class; the last bound holds because $\Gamma_F$-orbits have size at most $|G|$.

**Proposition 12 (recognising a product).** Let $A$ be an abelian surface over $F$. Then $A$ is isogenous
over $F$ to a product of elliptic curves iff $\operatorname{NS}(\bar A)^{\Gamma_F}$ contains a nonzero
isotropic class, and $A \cong E\times E'$ over $F$ iff it contains a hyperbolic plane, i.e. isotropic
$f_1,f_2$ with $f_1\cdot f_2 = 1$. (No Brauer obstruction intervenes:
$\operatorname{NS}(A) = \operatorname{NS}(\bar A)^{\Gamma_F}$ because $A$ has the rational point $0$.)

**Proposition 13 (irreducibility test).** On a K3 surface, if $D$ is effective, $C$ is an irreducible curve,
$C \ne D$ and $D\cdot C < 0$, then $C$ is a component of $D$. Consequently a class $E$ with $E^2 = -2$ and
$\deg E > 0$ (hence effective, by Riemann–Roch) is irreducible iff $E\cdot C \ge 0$ for every irreducible
curve $C \ne E$ of degree less than $\deg E$. **The exclusion $C \ne E$ is essential**: every curve
satisfies $E\cdot E = -2 < 0$.

### 17.5 The hypotheses, verified for the case at hand

Propositions 5, 7 and 10 have hypotheses that must be checked for the particular $\mathcal{N}$ in use. Here
$\mathcal{N}$ is the $96$-element set of section 10.

In [26]:
print("Proposition 5 needs: N is stable under Gamma-tilde")
S96 = set(orbit)
print("   Gamma-tilde permutes the 96 structures :",
      all(apply_perm(Pg[g], c) in S96 for g in gamma_tilde for c in orbit))
print()
print("Corollary 7 needs: every Stab n D is killed by 2")
print("   |Stab n D| for each structure :",
      sorted({len([g for g in S if g[1]==1]) for S in Stab}))
print("   every element of Stab n D squares to the identity :",
      all(all((2*x) % 4 == 0 for x in g[0]) for S in Stab for g in S if g[1]==1))
print("   => sqrt(a_i) must lie in Q(mu_8), and all four 4th roots in one quadratic extension of it")
print()
print("Corollary 10 needs: every Stab lies in D[2] x| Gal_8")
print("   exponents occurring anywhere in a stabiliser :",
      sorted({x for S in Stab for g in S for x in g[0]}),
      "-> all 2-torsion :", all(all(x % 2 == 0 for x in g[0]) for S in Stab for g in S))
print("   => no diagonal quartic with a real point has a stable structure in N")
print()
print("Proposition 4(5): the answer must not depend on the choice of fourth roots.")
Et = {1:{3:0,5:0,7:0,1:0}, -1:{3:1,5:2,7:3,1:0}, 4:{3:2,5:2,7:0,1:0}, -4:{3:3,5:0,7:3,1:0}}
def img(A, sgn):
    return frozenset((tuple((sgn*(Et[A[j]][m]-Et[A[0]][m])) % 4 for j in (1,2,3)), m) for m in (1,3,5,7))
h1 = {A for A in twists if any(img(A, 1) <= S for S in Stab)}
h2 = {A for A in twists if any(img(A,-1) <= S for S in Stab)}
print("   hits computed with delta   :", sorted(h1))
print("   hits computed with delta^-1:", sorted(h2))
print("   agree :", h1 == h2)

Proposition 5 needs: N is stable under Gamma-tilde
   Gamma-tilde permutes the 96 structures : True

Corollary 7 needs: every Stab n D is killed by 2
   |Stab n D| for each structure : [2]
   every element of Stab n D squares to the identity : True
   => sqrt(a_i) must lie in Q(mu_8), and all four 4th roots in one quadratic extension of it

Corollary 10 needs: every Stab lies in D[2] x| Gal_8
   exponents occurring anywhere in a stabiliser : [0, 2] -> all 2-torsion : True
   => no diagonal quartic with a real point has a stable structure in N

Proposition 4(5): the answer must not depend on the choice of fourth roots.


   hits computed with delta   : [(1, 1, 4, 4), (1, 4, 1, 4), (1, 4, 4, 1)]
   hits computed with delta^-1: [(1, 1, 4, 4), (1, 4, 1, 4), (1, 4, 4, 1)]
   agree : True


With these verified, the results of sections 11–16 read as follows in the language above.

- Section 11 computes, for the $64$ twists with $H_{\mathbf a}\cap D = 1$, which satisfy Proposition 5.
- Section 15 computes the full list $\{\operatorname{Stab}_{\tilde\Gamma}(N)\}$ and applies Corollaries 6
  and 7; the outcome is that $|H_{\mathbf a}| \in \{4,8\}$, giving the family
  $x^4+4y^4+d^2z^4+4d^2w^4=0$.
- Section 16 applies Proposition 9 and Corollary 10, giving the absence of real points.
- Section 9 is Proposition 11, section 13 is Proposition 3, section 14 is Proposition 12, and the
  correction discussed in section 12 is the exclusion clause of Proposition 13.

**What is not proved here.** Corollaries 7 and 10 are conditional on hypotheses about $\mathcal{N}$, and
$\mathcal{N}$ is the $96$-element set, not the set of *all* Kummer structures on $\bar X_{\mathbf 1}$.
Extending them to all diagonal quartics unconditionally would require knowing
$\operatorname{Stab}_{\tilde\Gamma}(N)$ for every Kummer structure $N$, which is a strictly larger
computation. Propositions 1–5, 8, 9 and 11–13 are unconditional.

## 18. How far the propositions can be pushed

Corollaries 7 and 10 were stated relative to the $96$-element $\mathcal{N}$ of section 10. It is natural to
ask whether they hold for *every* Kummer structure on $\bar X_{\mathbf 1}$, which would make them
unconditional statements about all diagonal quartics. This section settles that question — partly.

The tool is Proposition 11 applied not just to $H$ but to all of its subgroups. Two consequences:

**Proposition 14 (an $\Omega$-free test).** Suppose $X_{\mathbf a}$ has *some* Galois-stable Kummer
structure. Then for every subgroup $K \le H_{\mathbf a}$,
$$\dim \operatorname{Pic}(\bar X)^{K} \;\ge\; \left\lceil \tfrac{16}{|K|} \right\rceil + 1 .$$

*Proof.* Apply Proposition 11 to $K$: $\dim\operatorname{Pic}^K = \#(\Omega/K) + \dim\operatorname{NS}(\bar A)^K$.
Orbits of $K$ have size at most $|K|$, so $\#(\Omega/K) \ge \lceil 16/|K|\rceil$. And
$\operatorname{NS}(\bar A)^{H} \subseteq \operatorname{NS}(\bar A)^{K}$, while
$\operatorname{NS}(\bar A)^{H}$ contains an $H$-invariant ample class, so
$\dim\operatorname{NS}(\bar A)^K \ge 1$. $\;\blacksquare$

Note what this does *not* need: no degree bound, no enumeration of curves, no choice of $\mathcal{N}$. It
refers only to $\operatorname{Pic}\otimes\mathbb{Q}$ as an $H$-module.

**Proposition 15 (the $\Omega$ test).** If $X_{\mathbf a}$ has a Galois-stable Kummer structure with
underlying $H$-set $\Omega$, then the function
$$\nu(K) \;=\; \dim\operatorname{Pic}(\bar X)^{K} \;-\; \#(\Omega/K), \qquad K \le H,$$
satisfies $\nu(1) = 4$, $1 \le \nu(K) \le 4$ for all $K$, and $\nu(K') \le \nu(K)$ whenever
$K \subseteq K'$. (It is the invariant-dimension function of $\operatorname{NS}(\bar A)\otimes\mathbb{Q}$.)

Applying Proposition 15 means enumerating, for each $H$, every way $H$ could act on a $16$-element set —
that is, every multiset of transitive $H$-sets $H/K_i$ with $\sum [H:K_i] = 16$ — and asking whether any is
consistent.

### The Fermat quartic over $\mathbb{Q}$, unconditionally

For $\mathbf a = (1,1,1,1)$ the test is decisive and can be done by hand. Over the four characters of
$\Gamma = (\mathbb{Z}/2)^2$ the module $\operatorname{Pic}\otimes\mathbb{Q}$ has multiplicities
$(5,6,3,6)$. Orbits have size at most $4$, so $16$ points need at least $4$ orbits, while
$\dim\operatorname{Pic}^\Gamma - 1 = 4$ caps them at $4$: so $\Omega$ is exactly four free orbits and
$\mathbb{Q}[\Omega] = (4,4,4,4)$. The difference is $(1,2,-1,2)$, which has a negative multiplicity. No such
$\Omega$ exists.

In [27]:
# Theorem: the Fermat quartic over Q has NO Galois-stable Kummer structure -- of any degree,
# from any set of 16 curves whatsoever.  No search and no degree bound are involved.
G = [(m3,m5) for m3 in (0,1) for m5 in (0,1)]                  # Gal = <sigma_3> x <sigma_5>
Gmat = {(0,0): I20, (1,0): Gact[3], (0,1): Gact[5], (1,1): Gact[3]*Gact[5]}
def dimfixG(sub): 
    if not sub: return 20
    return matrix(QQ, sum([list((Gmat[g]-I20).rows()) for g in sub], [])).right_kernel().dimension()
subsG = [[], [(1,0)], [(0,1)], [(1,1)], [(1,0),(0,1)]]
print("dim Pic^K for the subgroups K of Gal(Q(mu_8)/Q):")
for sub in subsG:
    print("   K = %-22s |K| = %d,  dim Pic^K = %2d" %
          (str(sub) if sub else "trivial", 2^len(sub) if len(sub)<2 else 4, dimfixG(sub)))
print()
# enumerate every 16-element Gal-set and test Proposition 15
found = 0
for n1 in range(17):
  for n3 in range(9):
    for n5 in range(9):
      for n7 in range(9):
        for nr in range(5):
          if n1 + 2*(n3+n5+n7) + 4*nr != 16: continue
          # orbit counts of each subgroup K on Omega
          orb = {(): 16,
                 ((1,0),): n1 + n3 + 2*n5 + 2*n7 + 2*nr,
                 ((0,1),): n1 + 2*n3 + n5 + 2*n7 + 2*nr,
                 ((1,1),): n1 + 2*n3 + 2*n5 + n7 + 2*nr,
                 ((1,0),(0,1)): n1 + n3 + n5 + n7 + nr}
          nu = {}
          okc = True
          for sub, o in orb.items():
              v = dimfixG(list(sub)) - o
              if (v < 1 or v > 4) and sub != (): okc = False; break
              nu[sub] = v
          if okc and nu[()] == 4: found += 1
print("16-element Gal-sets consistent with Proposition 15 :", found)
print()
print("=> the Fermat quartic over Q carries NO Galois-stable Kummer structure.")
print("   This is unconditional: it does not depend on the 96 structures, nor on any degree bound.")

dim Pic^K for the subgroups K of Gal(Q(mu_8)/Q):
   K = trivial                |K| = 1,  dim Pic^K = 20
   K = [(1, 0)]               |K| = 2,  dim Pic^K = 11
   K = [(0, 1)]               |K| = 2,  dim Pic^K =  8
   K = [(1, 1)]               |K| = 2,  dim Pic^K = 11
   K = [(1, 0), (0, 1)]       |K| = 4,  dim Pic^K =  5



16-element Gal-sets consistent with Proposition 15 : 0

=> the Fermat quartic over Q carries NO Galois-stable Kummer structure.
   This is unconditional: it does not depend on the 96 structures, nor on any degree bound.


That removes the degree-$\le 12$ caveat which had been attached to this result throughout.

### The full sweep, and its limits

Doing the same for all of $\tilde\Gamma$ means enumerating every subgroup $H \le \tilde\Gamma$ with
$\operatorname{pr}_2(H) = \Gamma_8$. By the cocycle description, such an $H$ is determined by
$V = H\cap D$ together with a class in $Z^1(\Gamma_8, D/V)$, and there are $6765$ of them.

In [28]:
import itertools, time
Dv = list(itertools.product(range(4),repeat=3))
def addv(a,b): return tuple((a[i]+b[i]) % 4 for i in range(3))
def cloD(gens):
    S = {(0,0,0)}; q = [(0,0,0)]
    while q:
        a = q.pop()
        for g in gens:
            b = addv(a,g)
            if b not in S: S.add(b); q.append(b)
    return frozenset(S)
subsD = set()
for k in (0,1,2,3):
    for gens in itertools.combinations(Dv,k): subsD.add(cloD(gens))
print("subgroups of D = (Z/4)^3 :", len(subsD))

Hall = set()
for V in subsD:
    two = [c for c in Dv if tuple((2*c[i]) % 4 for i in range(3)) in V]
    for c3 in Dv:
        for c5 in two:
            c = {1:(0,0,0), 3:c3, 5:c5, 7:tuple((c3[i]-c5[i]) % 4 for i in range(3))}
            S = frozenset((addv(c[m],v), m) for m in (1,3,5,7) for v in V)
            if len(S) == 4*len(V): Hall.add(S)
print("subgroups H <= Gamma-tilde with pr_2(H) = Gal :", len(Hall))

subgroups of D = (Z/4)^3 : 129


subgroups H <= Gamma-tilde with pr_2(H) = Gal : 6765


In [29]:
# Proposition 14, applied with K of order 2 and 4
def mulG(g,h):
    (e,m),(f,l) = g,h
    return (tuple((e[i] + m*f[i]) % 4 for i in range(3)), (m*l) % 8)
Eid = ((0,0,0),1)
MATg = {g: act(gperm(g[0],g[1])) for g in gamma_tilde}
def dimfix(gens):
    gens = [g for g in gens if g != Eid]
    if not gens: return 20
    return matrix(QQ, sum([list((MATg[g]-I20).rows()) for g in gens],[])).right_kernel().dimension()
need = lambda k: (16 + k - 1)//k + 1

t0 = time.time()
invol = [g for g in gamma_tilde if g != Eid and mulG(g,g) == Eid]
bad2 = [g for g in invol if dimfix([g]) < need(2)]
sub4 = set()
for g in gamma_tilde:
    if mulG(g,g) != Eid and mulG(mulG(g,g), mulG(g,g)) == Eid:
        sub4.add(frozenset([Eid, g, mulG(g,g), mulG(g,mulG(g,g))]))
for a,b in itertools.combinations(invol,2):
    if mulG(a,b) == mulG(b,a) and mulG(a,b) in invol:
        sub4.add(frozenset([Eid,a,b,mulG(a,b)]))
bad4 = [K for K in sub4 if dimfix(list(K)) < need(4)]
print("involutions: %d, of which %d fail dim Pic^K >= 9" % (len(invol), len(bad2)))
print("order-4 subgroups: %d, of which %d fail dim Pic^K >= 5" % (len(sub4), len(bad4)))
banned = [frozenset([g]) for g in bad2] + list(bad4)
surv = [H for H in Hall if not any(B <= H for B in banned)]
print("H surviving Proposition 14 at orders 2 and 4 :", len(surv), " [%.0fs]" % (time.time()-t0))

involutions: 143, of which 9 fail dim Pic^K >= 9
order-4 subgroups: 1051, of which 320 fail dim Pic^K >= 5
H surviving Proposition 14 at orders 2 and 4 : 1056  [2s]


Continuing the pipeline externally — the remaining stages are the same tests, applied to every subgroup and
then with Proposition 15 — gives

| stage | remaining |
|---|---|
| all $H$ with $\operatorname{pr}_2(H) = \Gamma_8$ | $6765$ |
| Proposition 14 at orders $2$ and $4$ | $1056$ |
| reduce by $D$-conjugacy (Proposition 4(5)) | $288$ |
| Proposition 14 for every subgroup $K \le H$ | $282$ |
| Proposition 15 | $\mathbf{186}$ |

The twists $(1,1,4,4)$ and $(1,4,1,4)$ survive, as they must, and $(1,1,1,1)$ does not, as shown above.

**And here the method stops.** Among the $186$ survivors, $42$ have $H\cap D$ containing an element of
order $4$, and $168$ contain an element $(\delta,7)$ with $\delta$ of order $4$ — every parity pattern
occurs over $m=7$. So representation theory alone does **not** force $\sqrt{a_i}\in\mathbb{Q}(\mu_8)$, and
does **not** force the coefficients to share a sign. Corollaries 7 and 10 cannot be upgraded this way.

That is not a surprise in hindsight: every test in this section is a consequence of Proposition 11, which
sees only $\operatorname{Pic}\otimes\mathbb{Q}$ as an $H$-module. Corollaries 7 and 10 are statements about
*which* Kummer structures actually sit inside $\operatorname{Pic}$, and that is geometry, not character
theory. A concrete symptom: the mixed-sign twist $(1,-1,-1,1)$ passes every test in this section, yet the
exhaustive search of section 12 finds no configuration for it, and it is not among the hits over the $96$.

So the state of play is: Propositions 1–5, 8, 9, 11–15 and the theorem on the Fermat quartic over
$\mathbb{Q}$ are unconditional; Corollaries 7 and 10 remain theorems about the $96$-element $\mathcal{N}$.
Closing that gap requires either enumerating the Kummer structures on $\bar X_{\mathbf 1}$ beyond one
$\operatorname{Aut}$-orbit, or a genuinely geometric argument.

## 19. Corollary 10 is false

Section 16 concluded that a diagonal quartic with a real point cannot carry a Galois-stable Kummer
structure. That conclusion is correct for $\mathcal{N}$, the 96 structures of section 10, and the proof
given there is sound. It does not survive the passage to arbitrary Kummer structures.

To test it, take the 21 twists in $\{\pm1,\pm4\}$ with mixed signs whose Galois image survives every
representation-theoretic test of section 18 — these are exactly the ones Corollary 10 forbids and nothing
else rules out — and search for stable configurations of $(-2)$-curves of degree at most $D$.

**A warning about the filter, which matters here.** Sections 11–12 test irreducibility by asking whether
$E \cdot L \ge 0$ for every line $L \ne E$. That criterion is *exactly right* at degrees 2 and 3 — checked
below against a complete decomposition test — but it is **wrong at degree 4**, where a class can split as
conic $+$ conic with no line component. At degree 4 one needs the honest test: $E$ is reducible iff
$E = A + B$ with $A$ an irreducible curve of degree $1,2,3$ and $B$ effective. Since $E^2 = -2$ makes $E$
rigid, its effective divisor is unique, so this really does decide irreducibility.

In [30]:
# complete tables of irreducible classes in low degree
wq  = GP*hG
Q0f = -2*GP + matrix(QQ,[list(wq)]).transpose()*matrix(QQ,[list(wq)])
print("Q = -2*(form) + (.H)^2 is positive definite :", Q0f.is_positive_definite())
scf = lcm([QQ(x).denominator() for x in Q0f.list()]+[1]); Qif = matrix(ZZ, scf*Q0f)
def classes_ds(d, sq):
    out=[]
    for r in pari(Qif).qfminim(scf*(-2*sq + d^2), 10^7, 2)[2].sage().transpose().rows():
        for y in (vector(ZZ,r), -vector(ZZ,r)):
            if ip(y,y) == sq and ip(y,hG) == d: out.append(y)
    return out
EFF1 = {tuple(x) for x in lc}
A2   = classes_ds(2,-2)
IRR2 = [F for F in A2 if not any(tuple(F-L) in EFF1 for L in lc)]
EFF2 = {tuple(vector(ZZ,a)+vector(ZZ,b)) for a in EFF1 for b in EFF1} | {tuple(C) for C in IRR2}
A3   = classes_ds(3,-2) + classes_ds(3,0)         # twisted cubics and plane cubics
IRR3 = [F for F in A3 if not (any(tuple(F-L) in EFF2 for L in lc)
                              or any(tuple(F-C) in EFF1 for C in IRR2))]
EFF3 = {tuple(vector(ZZ,a)+vector(ZZ,b)) for a in EFF1 for b in EFF2} | {tuple(C) for C in IRR3}
print("degree-2 (-2)-classes %d, of which irreducible %d" % (len(A2), len(IRR2)))
print("degree-3 classes      %d, of which irreducible %d" % (len(A3), len(IRR3)))
def eff_rr(B): return ip(B,hG) > 0 and ip(B,B) >= -2      # Riemann-Roch certificate
def line_only(E): return all(ip(E,L) >= 0 for L in lc if tuple(L) != tuple(E))
def honest(E):
    d = ip(E,hG)
    if d <= 1: return True
    for A_ in lc:
        B = E - A_
        if ip(B,hG) > 0 and (tuple(B) in EFF1 or tuple(B) in EFF2 or tuple(B) in EFF3 or eff_rr(B)):
            return False
    if d >= 4:
        for A_ in IRR2:
            B = E - A_
            if ip(B,hG) > 0 and (tuple(B) in EFF1 or tuple(B) in EFF2 or eff_rr(B)): return False
        for A_ in IRR3:
            B = E - A_
            if ip(B,hG) > 0 and (tuple(B) in EFF1 or eff_rr(B)): return False
    return True
for d in (2,3):
    Cd = [F for F in classes_ds(d,-2)]
    a = {tuple(F) for F in Cd if line_only(F)}; b = {tuple(F) for F in Cd if honest(F)}
    print("   degree %d: line-only filter and honest test agree : %s  (%d classes kept)"
          % (d, a == b, len(a)))

Q = -2*(form) + (.H)^2 is positive definite : True


degree-2 (-2)-classes 656, of which irreducible 320
degree-3 classes      12192, of which irreducible 1152


   degree 2: line-only filter and honest test agree : True  (320 classes kept)


   degree 3: line-only filter and honest test agree : True  (1152 classes kept)


So the cheap filter was safe for everything done at degrees $\le 3$, and the searches of sections 11 and 12
stand. At degree 4 the honest test is needed — and with it, the following turns up.

In [31]:
import time
A = (1,-1,4,-4); Db = 4
M = {m: act_matrix(tperm(A,m)) for m in (3,5,7)}; M[1] = identity_matrix(ZZ,20)
Av = (matrix(QQ,M[1])+M[3]+M[5]+M[7])/4
BG = matrix(QQ, matrix(QQ,Av).row_module(ZZ).basis())
BW = matrix(QQ, matrix(QQ,E20-Av).row_module(ZZ).basis()); rg = BG.nrows()
GrG = BG*GP*BG.transpose()
cG  = vector(QQ,[ip(vector(QQ,BG.row(i)), hG) for i in range(rg)])
QG0 = -2*GrG + matrix(QQ,[[cG[i]*cG[j] for j in range(rg)] for i in range(rg)])
sG  = lcm([QQ(x).denominator() for x in QG0.list()]+[1]); QG = matrix(ZZ, sG*QG0)
cand = set()
for s in (1,2,4):
    e1t = QQ(-s)/2; Xis = [tuple(x) for x in shorts(BW, QQ(-2)+QQ(s)/2)]
    for r in pari(QG).qfminim(sG*(-2*e1t + Db*Db), 10^7, 2)[2].sage().transpose().rows():
        y = vector(ZZ,r)
        if y*GrG*y != e1t: continue
        dg = y*cG
        if   1 <= dg <= Db: pass
        elif 1 <= -dg <= Db: y = -y
        else: continue
        v = vector(QQ,y)*BG
        for x in Xis:
            E = tuple(v[i]+x[i] for i in range(20))
            if all(c in ZZ for c in E): cand.add(tuple(ZZ(c) for c in E))
allC = [vector(ZZ,c) for c in cand]
C = [E for E in allC if honest(E)]
print("x^4 - y^4 + 4z^4 - 4w^4 = 0 :  %d classes of degree <= 4, %d genuinely irreducible"
      % (len(allC), len(C)))
key = {tuple(c):i for i,c in enumerate(C)}
seen=set(); orbs=[]
for i,c in enumerate(C):
    if i in seen: continue
    img = [key.get(tuple(c*M[m])) for m in (1,3,5,7)]
    if any(t is None for t in img): continue
    o = sorted(set(img)); seen |= set(o)
    if all(ip(C[a_],C[b_])==0 for a_ in o for b_ in o if a_<b_): orbs.append(o)
mm = len(orbs)
R = matrix(ZZ,[C[o[0]] for o in orbs]); PR = {q: R*GP*(R*M[q]).transpose() for q in (1,3,5,7)}
comp = [[all(PR[q][i,j]==0 for q in (1,3,5,7)) for j in range(mm)] for i in range(mm)]
sols=[]
def rec(start, chosen, tot):
    if tot == 16:
        Ssum = sum(C[t] for o in chosen for t in orbs[o])
        if all(x % 2 == 0 for x in Ssum): sols.append(list(chosen))
        return
    if tot > 16 or len(chosen) >= rg-1: return
    for i in range(start, mm):
        if len(orbs[i])+tot > 16: continue
        if all(comp[i][j] for j in chosen):
            chosen.append(i); rec(i+1, chosen, tot+len(orbs[i])); chosen.pop()
rec(0,[],0)
print("Galois orbits: %d ;  GALOIS-STABLE KUMMER STRUCTURES: %d" % (mm, len(sols)))

x^4 - y^4 + 4z^4 - 4w^4 = 0 :  7304 classes of degree <= 4, 296 genuinely irreducible
Galois orbits: 118 ;  GALOIS-STABLE KUMMER STRUCTURES: 16


In [32]:
# full independent verification of one of them
V = [C[t] for o in sols[0] for t in orbs[o]]
PicZ = (matrix(QQ, [list(toPic(vector(QQ,linecoord[a]))) for a in range(48)])).row_module(ZZ)
print("16 distinct classes          :", len({tuple(x) for x in V}) == 16)
print("every E^2 = -2               :", all(ip(E,E) == -2 for E in V))
print("pairwise disjoint            :", all(ip(V[i],V[j])==0 for i in range(16) for j in range(i+1,16)))
print("sum 2-divisible in Pic       :", (sum(V)/2) in PicZ)
print("every member irreducible     :", all(honest(E) for E in V))
print("degrees                      :", sorted(ip(E,hG) for E in V))
Vs = {tuple(x) for x in V}
print("stable under twisted sigma_3 :", {tuple(E*M[3]) for E in V} == Vs)
print("stable under twisted sigma_5 :", {tuple(E*M[5]) for E in V} == Vs)
print("stable under twisted sigma_7 :", {tuple(E*M[7]) for E in V} == Vs)
grp = [M[1],M[3],M[5],M[7]]; seen=set(); sz=[]
for E in V:
    if tuple(E) in seen: continue
    o = {tuple(E*g) for g in grp}; seen |= o; sz.append(len(o))
rk = matrix(QQ, sum([list((M[q]-identity_matrix(QQ,20)).rows()) for q in (3,5,7)],[])).right_kernel().dimension()
print("orbit sizes                  :", sorted(sz), "->", len(sz), "orbits")
print("rank identity                :", len(sz), "+", rk-len(sz), "=", rk)
print("Galois-fixed curves          :", sum(1 for x in sz if x == 1))
print()
print("and the surface has real points:  1 - 1 + 4 - 4 =", 1-1+4-4, " at (1:1:1:1)")

16 distinct classes          : True
every E^2 = -2               : True
pairwise disjoint            : True
sum 2-divisible in Pic       : True


every member irreducible     : True
degrees                      : [1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 4]
stable under twisted sigma_3 : True
stable under twisted sigma_5 : True
stable under twisted sigma_7 : True
orbit sizes                  : [2, 2, 2, 2, 4, 4] -> 6 orbits
rank identity                : 6 + 3 = 9
Galois-fixed curves          : 0

and the surface has real points:  1 - 1 + 4 - 4 = 0  at (1:1:1:1)


### What this changes

$$x^4 - y^4 + 4z^4 - 4w^4 = 0$$
has real points and **is** a Kummer surface over $\mathbb{Q}$. Its stable structure consists of four lines,
ten conics and two curves of degree 4 — and it is precisely those two quartic curves that put it outside
$\mathcal{N}$, which is why sections 15 and 16 could not see it. The same holds for the six coordinate
permutations of this twist. Note $abcd = 16$ is a fourth power here, as it was for $x^4+y^4+4z^4+4w^4$.

So:

- **Corollary 10 is false** as a statement about diagonal quartics. It is true as stated, about $\mathcal{N}$.
- The infinite family of section 15 is not the complete list of diagonal quartics that are Kummer surfaces
  over $\mathbb{Q}$; it is the complete list of those detected by $\mathcal{N}$.
- Corollary 7 is presumably in the same position. Nothing here refutes it, but nothing supports it beyond
  $\mathcal{N}$ either.
- Searches at degree bounds 2 and 3 found nothing for any of the 21 mixed-sign candidates, and the filter
  used there is provably exact, so those negatives are sound. The counterexample appears at degree exactly 4.

Sweeping all 21 mixed-sign candidates at degree bound 4 with the honest filter (run externally) gives
exactly six hits, and they are the six coordinate permutations of one surface:

| twist | rank $\operatorname{Pic}^\Gamma$ | classes | orbits | configurations |
|---|---|---|---|---|
| $(1,1,4,4)$ (control) | 9 | 7808 | 156 | 16 |
| $(1,-1,4,-4)$ and its 5 permutations | 9 | 7304 | 118 | **16** each |
| the other 15 mixed-sign candidates | 8 or 9 | 3024-6432 | 108-136 | 0 |

The general lesson is the one already visible in section 12, now with a second instance: results proved
about a *chosen* family of Kummer structures do not transfer to all of them, and the irreducibility test has
to be matched to the degree it is applied at.

## 20. Three questions about the counterexample

### Does it have rational points?

Yes, and easily: $(1:1:1:1)$, $(1:1:0:0)$ and $(0:0:1:1)$ all lie on $x^4-y^4+4z^4-4w^4=0$. That is a
sharp contrast with the family of section 15, which has no points over $\mathbb{R}$ and none over
$\mathbb{Q}_2$.

There is no tension with the fact that the associated $V$ is a *nontrivial* torsor. A rational point of
$\operatorname{Kum}(V)$ away from the exceptional curves is a Galois-stable pair $\{v,-v\}$, i.e. a point
with $\sigma(v) \in \{v,-v\}$ for every $\sigma$; that is much weaker than $\sigma(v) = v$. So
$\operatorname{Kum}(V)(\mathbb{Q}) \neq \varnothing$ does not force $V(\mathbb{Q}) \neq \varnothing$.

### Over $\mathbb{R}$, is it the Kummer surface of an honest abelian surface?

Over $\mathbb{R}$ the acting group is only $\langle c \rangle$ with $c$ complex conjugation, which is far
weaker than $\operatorname{Gal}(\mathbb{Q}(\mu_8)/\mathbb{Q})$, so there is much more room for a stable
structure. By Proposition 3 the question is whether some conjugation-stable Kummer structure has a
$c$-**fixed** curve. The rank identity leaves the possibility open: $\operatorname{rank}\operatorname{Pic}^{c} = 11 = (8 + f/2) + r$
with $r \ge 1$ forces only $f \in \{0,2,4\}$.

In [33]:
print("x^4 - y^4 + 4z^4 - 4w^4 = 0 :  1-1+4-4 =", 1-1+4-4, " so (1:1:1:1) is a rational point")
print()
# the 16 structures found over Q, now seen only through complex conjugation
M7 = M[7]; I20z = identity_matrix(ZZ,20)
print("complex conjugation is an involution on Pic :", M7*M7 == I20z)
print("rank Pic^<conj> =", matrix(QQ,(M7-I20z).rows()).right_kernel().dimension())
from collections import Counter
prof = Counter()
for so in sols:
    V0 = [C[t] for o in so for t in orbs[o]]
    seen=set(); sz=[]
    for E in V0:
        if tuple(E) in seen: continue
        o = {tuple(E), tuple(E*M7)}; seen |= o; sz.append(len(o))
    prof[(tuple(sorted(sz)), sum(1 for x in sz if x==1))] += 1
for (sz,f),k in sorted(prof.items()):
    print("   %2d structures: conj-orbit sizes %s -> %d orbits, %d FIXED curves" % (k, list(sz), len(sz), f))

x^4 - y^4 + 4z^4 - 4w^4 = 0 :  1-1+4-4 = 0  so (1:1:1:1) is a rational point

complex conjugation is an involution on Pic : True
rank Pic^<conj> = 11
   16 structures: conj-orbit sizes [2, 2, 2, 2, 2, 2, 2, 2] -> 8 orbits, 0 FIXED curves


So the structures we already have give no fixed curve. But there are other conjugation-stable structures,
and they have to be looked at too. Decompose $\operatorname{Pic}$ into the $\pm1$ eigenlattices of $c$ and
enumerate: a $c$-fixed curve lies in the $+$ part, a swapped pair contributes $E_1 \in {+}$ with
$E_1^2 = -1$ and $\xi \in {-}$ with $\xi^2 = -1$.

In [34]:
import time
Avc = (matrix(QQ,I20z) + M7)/2
BPc = matrix(QQ, matrix(QQ,Avc).row_module(ZZ).basis())
BMc = matrix(QQ, matrix(QQ,E20-Avc).row_module(ZZ).basis())
GrP = BPc*GP*BPc.transpose()
cPv = vector(QQ,[ip(vector(QQ,BPc.row(i)),hG) for i in range(BPc.nrows())])
QP0 = -2*GrP + matrix(QQ,[[cPv[i]*cPv[j] for j in range(BPc.nrows())] for i in range(BPc.nrows())])
sPc = lcm([QQ(x).denominator() for x in QP0.list()]+[1]); QPi = matrix(ZZ, sPc*QP0)
print("invariant lattice rank %d, anti-invariant rank %d, Q positive definite %s"
      % (BPc.nrows(), BMc.nrows(), QP0.is_positive_definite()))
Dc = 4; buckets = {}
for ss in (1,2):                       # ss=2: c-fixed curve;  ss=1: member of a swapped pair
    e1t = QQ(-ss)
    Xis = [tuple(x) for x in shorts(BMc, QQ(-2)-e1t)]
    got = set()
    for r in pari(QPi).qfminim(sPc*(-2*e1t + Dc*Dc), 10^7, 2)[2].sage().transpose().rows():
        y = vector(ZZ,r)
        if y*GrP*y != e1t: continue
        dg = y*cPv
        if   1 <= dg <= Dc: pass
        elif 1 <= -dg <= Dc: y = -y
        else: continue
        v = vector(QQ,y)*BPc
        for x in Xis:
            E = tuple(v[i]+x[i] for i in range(20))
            if all(c_ in ZZ for c_ in E): got.add(tuple(ZZ(c_) for c_ in E))
    buckets[ss] = [vector(ZZ,c_) for c_ in got if honest(vector(ZZ,c_))]
FIXED  = [E for E in buckets[2] if tuple(E*M7) == tuple(E)]
SWAP   = [E for E in buckets[1] if tuple(E*M7) != tuple(E)]
print("conjugation-FIXED irreducible (-2)-curves of degree <= %d : %d" % (Dc, len(FIXED)))
print("swapped ones                                            : %d" % len(SWAP))

invariant lattice rank 11, anti-invariant rank 9, Q positive definite True


conjugation-FIXED irreducible (-2)-curves of degree <= 4 : 64
swapped ones                                            : 632


In [35]:
# assemble: f fixed curves plus (16-f)/2 conjugate pairs, all mutually disjoint, sum 2-divisible
seenp=set(); pairs=[]
for E in SWAP:
    t=tuple(E)
    if t in seenp: continue
    Ec=E*M7; seenp |= {t, tuple(Ec)}
    if ip(E,Ec)==0: pairs.append((E,Ec))
items = [("f",[E]) for E in FIXED] + [("p",[a,b]) for (a,b) in pairs]
N = len(items)
compat=[[all(ip(a,b)==0 for a in items[i][1] for b in items[j][1]) for j in range(N)] for i in range(N)]
PicZ2=(matrix(QQ,[list(toPic(vector(QQ,linecoord[a]))) for a in range(48)])).row_module(ZZ)
print("items (fixed curves + conjugate pairs) : %d" % N)
def assemble(require_fixed):
    out=[]
    def rec(st, ch, tot, nf):
        if tot==16:
            if nf >= (1 if require_fixed else 0):
                S=sum(sum(items[i][1]) for i in ch)
                if all(x%2==0 for x in S) and (S/2) in PicZ2: out.append(list(ch))
            return
        if tot>16: return
        for i in range(st,N):
            wgt=len(items[i][1])
            if tot+wgt>16 or nf+(1 if items[i][0]=="f" else 0) > 4: continue
            if all(compat[i][j] for j in ch):
                ch.append(i); rec(i+1,ch,tot+wgt,nf+(1 if items[i][0]=="f" else 0)); ch.pop()
    rec(0,[],0,0); return out
t0=time.time(); allc = assemble(False); t1=time.time()
withf = [ch for ch in allc if any(items[i][0]=="f" for i in ch)]
print("conjugation-stable Kummer structures (degree <= 4) : %d   [%.0fs]" % (len(allc), t1-t0))
print("   with at least one conjugation-FIXED curve      : %d" % len(withf))
print("   distribution of fixed-curve counts             :",
      dict(Counter(sum(1 for i in ch if items[i][0]=='f') for ch in allc)))
print()
print("=> over R the surface is Kum(a nontrivial torsor), not Kum(an abelian surface),")
print("   for every Kummer structure whose curves have degree at most 4.")

items (fixed curves + conjugate pairs) : 380


conjugation-stable Kummer structures (degree <= 4) : 32   [14s]
   with at least one conjugation-FIXED curve      : 0
   distribution of fixed-curve counts             : {0: 32}

=> over R the surface is Kum(a nontrivial torsor), not Kum(an abelian surface),
   for every Kummer structure whose curves have degree at most 4.


The control matters here: dropping the fixed-curve requirement finds $32$ conjugation-stable structures —
the $16$ that are stable over all of $\mathbb{Q}$, plus $16$ more that are stable only under conjugation —
and every one of them has $f = 0$. So the search is finding things, and the answer is a genuine negative
rather than an empty search.

### Does the flipped family work?

Section 15 produced $x^4+4y^4+d^2z^4+4d^2w^4=0$. Since $(1,4,-1,-4)$ is one of the six twists of section
19, it is natural to ask whether the sign-flipped family $x^4+4y^4-d^2z^4-4d^2w^4=0$ works for all $d$ as
well. It does not: only $d=1$.

In [36]:
def galois_mats(A, p=None):
    "matrices of the Galois image; p=None when the fourth roots already lie in Q(mu_8)"
    if p is None:
        Et={1:{3:0,5:0,7:0,1:0},-1:{3:1,5:2,7:3,1:0},4:{3:2,5:2,7:0,1:0},-4:{3:3,5:0,7:3,1:0}}
        return [act_matrix(compose(p_delta([(Et[A[j]][mm]-Et[A[0]][mm])%4 for j in range(4)]),
                                   p_sigma(mm))) for mm in (1,3,5,7)]
    Fc.<zc>=CyclotomicField(8); Rc.<Xc>=Fc[]
    Lc.<spc>=Fc.extension(Xc^2-p); Mc.<tc>=Lc.absolute_field(); emb=Mc.structure()[1]
    zM,spM=emb(zc),emb(spc); iM=zM^2
    root={1:Mc(1),-1:zM,4:zM+zM^-1,-4:1+zM^2,
          p^2:spM,-p^2:zM*spM,4*p^2:(zM+zM^-1)*spM,-4*p^2:(1+zM^2)*spM}
    for v_,u_ in root.items(): assert u_^4==v_
    def dlog(x):
        for e_ in range(4):
            if x==iM^e_: return e_
    out=[]
    for aut in Mc.automorphisms():
        mm=[q for q in (1,3,5,7) if aut(zM)==zM^q][0]
        e_=[(dlog(aut(root[A[j]])/root[A[j]])-dlog(aut(root[A[0]])/root[A[0]]))%4 for j in range(4)]
        out.append(act_matrix(compose(p_delta(e_), p_sigma(mm))))
    return out

def search_twist(A, mats, Db=4, tag=""):
    Nm=len(mats)
    Avm=sum(matrix(QQ,Mx) for Mx in mats)/Nm
    BGm=matrix(QQ,matrix(QQ,Avm).row_module(ZZ).basis())
    BWm=matrix(QQ,matrix(QQ,E20-Avm).row_module(ZZ).basis()); rgm=BGm.nrows()
    GrM=BGm*GP*BGm.transpose(); cM=vector(QQ,[ip(vector(QQ,BGm.row(i)),hG) for i in range(rgm)])
    QM0=-2*GrM+matrix(QQ,[[cM[i]*cM[j] for j in range(rgm)] for i in range(rgm)])
    if not QM0.is_positive_definite(): print("   %s : not definite" % tag); return None
    sM=lcm([QQ(x).denominator() for x in QM0.list()]+[1]); QMi=matrix(ZZ,sM*QM0)
    cnd=set()
    for ss in [k for k in (1,2,4,8) if k<=Nm and Nm%k==0 and Nm//k<=16]:
        e1t=QQ(-2*ss)/Nm
        Xis=[tuple(x) for x in shorts(BWm, QQ(-2)-e1t)]
        for r in pari(QMi).qfminim(sM*(-2*e1t+Db*Db),10^7,2)[2].sage().transpose().rows():
            y=vector(ZZ,r)
            if y*GrM*y!=e1t: continue
            dg=y*cM
            if 1<=dg<=Db: pass
            elif 1<=-dg<=Db: y=-y
            else: continue
            v=vector(QQ,y)*BGm
            for x in Xis:
                E=tuple(v[i]+x[i] for i in range(20))
                if all(c_ in ZZ for c_ in E): cnd.add(tuple(ZZ(c_) for c_ in E))
    Cm=[E for E in (vector(ZZ,c_) for c_ in cnd) if honest(E)]
    keym={tuple(c_):i for i,c_ in enumerate(Cm)}; sn=set(); ob=[]
    for i,c_ in enumerate(Cm):
        if i in sn: continue
        img=[keym.get(tuple(c_*Mx)) for Mx in mats]
        if any(t is None for t in img): continue
        o=sorted(set(img)); sn|=set(o)
        if all(ip(Cm[a_],Cm[b_])==0 for a_ in o for b_ in o if a_<b_): ob.append(o)
    mo=len(ob)
    if mo==0:
        print("   %-20s rank %d, %5d irred classes, 0 orbits -> 0" % (tag,rgm,len(Cm))); return 0
    Rm=matrix(ZZ,[Cm[o[0]] for o in ob]); PRm={k:Rm*GP*(Rm*mats[k]).transpose() for k in range(Nm)}
    cp=[[all(PRm[k][i,j]==0 for k in range(Nm)) for j in range(mo)] for i in range(mo)]
    ss_=[]
    def rc(st,ch,tot):
        if tot==16:
            S=sum(Cm[t] for o in ch for t in ob[o])
            if all(x%2==0 for x in S): ss_.append(1)
            return
        if tot>16 or len(ch)>=rgm: return
        for i in range(st,mo):
            if len(ob[i])+tot>16: continue
            if all(cp[i][j] for j in ch):
                ch.append(i); rc(i+1,ch,tot+len(ob[i])); ch.pop()
    rc(0,[],0)
    print("   %-20s rank %d, %5d irred classes, %3d orbits -> %d configurations"
          % (tag,rgm,len(Cm),mo,len(ss_)))
    return len(ss_)

search_twist((1,4,-1,-4), galois_mats((1,4,-1,-4)), 4, "(1,4,-1,-4)  d=1")
search_twist((1,4,-9,-36), galois_mats((1,4,-9,-36),3), 4, "(1,4,-9,-36) d=3")
search_twist((1,4,9,36),   galois_mats((1,4,9,36),3),   4, "(1,4,9,36)   d=3")

   (1,4,-1,-4)  d=1     rank 9,   296 irred classes, 118 orbits -> 16 configurations


   (1,4,-9,-36) d=3     rank 7,    64 irred classes,  32 orbits -> 0 configurations


   (1,4,9,36)   d=3     rank 7,   144 irred classes,  40 orbits -> 8 configurations


8

### The family $x^4 - y^4 + c^2z^4 - c^2w^4 = 0$

The counterexample is the case $c = 2$ of this family, and $c=1$ is the classical surface
$x^4 + z^4 = y^4 + w^4$ of equal sums of two fourth powers. Every member has $abcd = c^4$, a fourth power,
and every member has the obvious rational points $(1:1:0:0)$, $(0:0:1:1)$, $(1:1:1:1)$. So it is a natural
family to ask about. Here $c=1$ repeats a twist already covered — it is one of the 21 mixed-sign
candidates — and serves as a consistency check.

In [37]:
for c_, p_ in [(1,None),(2,None),(3,3)]:
    A_ = [1,-1,c_^2,-c_^2]
    red = []
    for v_ in A_:
        y_ = ZZ(abs(v_)); sg = 1 if v_ > 0 else -1
        for q_ in (2,3,5,7):
            while y_ % q_^4 == 0: y_ //= q_^4
        red.append(sg*y_)
    search_twist(tuple(red), galois_mats(tuple(red), p_), 4, "c=%d  %s" % (c_, tuple(red)))

   c=1  (1, -1, 1, -1)  rank 9,   520 irred classes, 136 orbits -> 0 configurations


   c=2  (1, -1, 4, -4)  rank 9,   296 irred classes, 118 orbits -> 16 configurations


   c=3  (1, -1, 9, -9)  rank 7,    72 irred classes,  26 orbits -> 0 configurations


The last line is a useful cross-check rather than a new result: this search, which enumerates $(-2)$-classes
from scratch, recovers the **8** structures that section 15 found for $x^4+4y^4+9z^4+36w^4=0$ by the
completely different route of stabilisers inside $\tilde\Gamma$. Two independent methods, same answer.
(The same computation with $p=5$ gives the same pattern: $0$ for the flipped twist, $8$ for the unflipped.)

So the sign-flipped family stops at $d=1$ — with the usual caveat, which here is a serious one: the search
is bounded by degree $4$, and $(1,-1,4,-4)$ itself only becomes visible *at* degree $4$. A flipped example
at $d=3$ could be waiting at degree $5$ or $6$.

### Summary of this section

- $x^4-y^4+4z^4-4w^4=0$ has rational points, unlike the family of section 15.
- Over $\mathbb{R}$ it is still $\operatorname{Kum}$ of a nontrivial torsor: no conjugation-stable Kummer
  structure of degree $\le 4$ has a fixed curve, though $64$ fixed $(-2)$-curves exist in that range.
- The sign-flipped family works only for $d=1$, up to degree $4$.
- The family $x^4-y^4+c^2z^4-c^2w^4=0$ works only for $c=2$, up to degree $4$: $c=1,3,5,6$ all give nothing
  ($c=5$ and $c=6$ checked separately). In particular the classical surface $x^4+z^4=y^4+w^4$ is *not* a
  Kummer surface over $\mathbb{Q}$ by any structure of degree $\le 4$.

- A third family, $x^4-4y^4+d^2z^4-4d^2w^4=0$, contains our example at $d=2$ (since
  $(1,-4,4,-16)\equiv(1,-4,4,-1)$ modulo fourth powers). Run externally at degree bound 4 it gives
  configurations only at $d=2$: $d=1,3,5,6$ all return $0$. Their invariant lattices have
  $\operatorname{rank}\operatorname{Pic}^\Gamma = 5$, the minimum, which by Proposition 11 forces four free
  orbits of four — a very rigid demand — whereas the successful twist has rank $9$.

So $x^4-y^4+4z^4-4w^4=0$ and its six coordinate permutations resist generalisation in all three directions
tried, each one-parameter family meeting the set of Kummer surfaces in that single point. Whether that is a
real feature or an artifact of the degree-4 ceiling is exactly the question the ceiling prevents us from
answering — and it is worth remembering that this surface is itself invisible below degree 4.

## 21. Summary

| surface | field | Kummer surface there? |
|---|---|---|
| $x^4+y^4+z^4+w^4$ | $\mathbb{Q}(\mu_8)$ | yes (Mizukami) |
| $x^4+y^4+z^4+w^4$ | $\mathbb{Q}(\sqrt{-2})$ | **yes**, 24 structures |
| $x^4+y^4+z^4+w^4$ | $\mathbb{Q}(\sqrt{2})$ | **yes**, 24 structures |
| $x^4+y^4+z^4+w^4$ | $\mathbb{Q}(i)$ | no — excluded by the rank count alone |
| $x^4+y^4+z^4+w^4$ | $\mathbb{Q}$ | no |
| $x^4+y^4+4z^4+4w^4$ | $\mathbb{Q}$ | **yes**, 16 structures |
| $x^4+4y^4+d^2z^4+4d^2w^4$, any $d$ | $\mathbb{Q}$ | **yes** — an infinite family (section 15) |
| $x^4-y^4+4z^4-4w^4$ | $\mathbb{Q}$ | **yes**, and it has real points (section 19) |

So the answer to the question in the title is: the Fermat quartic is a Kummer surface over each of
$\mathbb{Q}(\sqrt2)$ and $\mathbb{Q}(\sqrt{-2})$, and suitable diagonal twists of it are Kummer surfaces
over $\mathbb{Q}$ — infinitely many of them, namely $x^4+4y^4+d^2z^4+4d^2w^4 = 0$ for every positive
integer $d$. All of these facts were unexpected; the field of definition of Mizukami's isomorphism is not
as rigid as it looks.

Sections 15 and 16 draw two restrictions from the stabilisers of the 96 structures: that
$\sqrt{a_i}\in\mathbb{Q}(\mu_8)$, and that all the coefficients share a sign. **Both are theorems about
$\mathcal{N}$ only, and the second is false in general** — section 19 exhibits
$x^4-y^4+4z^4-4w^4=0$, which has real points and is a Kummer surface over $\mathbb{Q}$ by a structure
containing two quartic curves. In every case found, stable or not, Galois acts on the 16 curves without a
fixed point, so the object underneath is a nontrivial torsor.

Section 14 refines this: the abelian surface underneath is, for several of the configurations, an actual
product $E\times E'$ of elliptic curves over the ground field. The one thing that does **not** descend is
the origin. In every configuration found, Galois permutes the 16
curves without fixing any, so what sits on the other side is $\operatorname{Kum}(V)$ for a nontrivial
2-covering torsor $V$, not $\operatorname{Kum}(A)$ for an abelian surface, and in particular not
$\operatorname{Kum}(E\times E')$. The classical dictionary "points on the Kummer surface $=$ points on twists
of $E\times E'$ modulo $\pm$" therefore still does not survive descent in the form one would want.

**What is proved and what is not.**

- The positive results are rigorous: an explicit set of 16 pairwise disjoint irreducible curves with
  2-divisible sum, checked to be permuted by the relevant Galois group. Nikulin's criterion then applies.
- "No over $\mathbb{Q}(i)$" is rigorous and unconditional: it follows from the rank identity of section 9,
  which needs no search.
- "No over $\mathbb{Q}$ for the Fermat quartic" is **unconditional**, by the character argument of section
  18; the degree-$\le 12$ enumeration in the companion descent notebook is superseded by it.
- The restrictions in sections 15 and 16 are rigorous statements about $\mathcal{N}$, the 96 structures of
  section 10. They are **not** statements about all Kummer structures: section 19 disproves the section-16
  one outright. Treat every "no" in sections 11, 15 and 16 as "no, within $\mathcal{N}$".
- The statement that these particular structures give torsors rather than abelian surfaces is rigorous for
  the structures exhibited. It does not by itself exclude some *other* Kummer structure, outside the 96
  examined here, having a Galois-fixed curve.

## References

- E. Ieronymou, A. N. Skorobogatov, Yu. G. Zarhin, *On the Brauer group of diagonal quartic surfaces*, with
  an appendix by Sir Peter Swinnerton-Dyer, J. London Math. Soc. **83** (2011) 659–672,
  [arXiv:0912.2865](https://arxiv.org/abs/0912.2865).
- R. G. E. Pinch, H. P. F. Swinnerton-Dyer, *Arithmetic of diagonal quartic surfaces I*, in: L-functions and
  Arithmetic, LMS Lecture Notes **153** (1991) 317–338 — the 48 lines generate $\operatorname{Pic}$.
- V. V. Nikulin, *On Kummer surfaces*, Izv. Akad. Nauk SSSR Ser. Mat. **39** (1975) 278–293.
- A. N. Skorobogatov, Yu. G. Zarhin, *The Brauer group of Kummer surfaces and torsion of elliptic curves*,
  J. reine angew. Math. **666** (2012) 115–140 — Kummer surfaces of torsors.